# X-CLIP base/32 — DIMER E2E video-classification adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/xclip-video-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/xclip-video-classification-pipeline/blob/main/tutorials/xclip_video_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Fxclip--base--patch32-ffcc4d?style=flat)](https://huggingface.co/microsoft/xclip-base-patch32) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FVideoX-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/VideoX/tree/master/X-CLIP) [![arXiv](https://img.shields.io/badge/arXiv-2208.02816-b31b1b.svg)](https://arxiv.org/abs/2208.02816)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** zero-shot video classification — one 8-frame clip plus 2–32 free-text class names → a ranking of those names with a softmax over them — and bounded supervised fine-tuning of the fusion head on labelled clips of a closed label set, using the pinned `microsoft/xclip-base-patch32` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/xclip_video_classification_pipeline/`, at revision `1a3e19692585`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `a2e27a78a2b5d802e894b8a1ef14f3a8ce490963` (~790 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (nothing is installed into the notebook's own Python, so no restart is needed and Run all completes in one pass), stages and digest-verifies the pinned `microsoft/xclip-base-patch32` snapshot (a 786 MB `model.safetensors`; no pickle is opened anywhere), fetches the first row group of the HMDB51 test shard from the Hugging Face Hub at an immutable revision with one HTTPS range request (about 113 MB; refused on any SHA-256 or byte-total mismatch), decodes the 300 clips of the ten sample classes into 8 uniformly spaced frames each with PyAV, validates the records and splits them by source video into 181 / 53 / 66, ranks five drawn clips through the inference contract with a combined input manifest and a rejection probe, scores the frozen model over the 66 held-out clips (top-1 and top-3 accuracy, macro recall and F1) beside a chance and a majority-label baseline, runs a bounded fine-tuning of the fusion head on cached tower features with validation-accuracy epoch selection, scores the held-out clips again, re-runs six held-out clips and the five drawn clips with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify ranking parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). On a Tesla T4 the default path took about 2 minutes of cell time (eight epochs 12 s, frozen scoring of 66 clips 3 s); a CUDA runtime is used automatically when present, and the path is practical on CPU too (the build venv decoded the 300 clips in about 70 s, scored the test split in 10 s and ran the eight epochs in about 60 s).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and either set `BYOD_PATH` to a zip or folder in the runtime (Colab, Kaggle or Jupyter) or leave it empty to upload one zip in Colab, then choose **Run after** from that cell (it first puts the model back to the pinned base) to supply clips (AVI, MP4, MOV, MKV, WebM, GIF or WebP; at least 8 frames each) plus a `labels.csv` (`file`, `label`, optional `id` and `source`; one row per clip — at least **24 clips for two labels** (12 per label), 27 for three, 24 for four (`min_byod_records(n_labels)`); a `source` keeps all its clips, of every label, in one split). The records pass through the same validation, source-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the HMDB51 sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

X-CLIP base/32 is a video–text model: a CLIP ViT-B/32 frame encoder with cross-frame attention embeds each of the 8 frames of a clip, a one-layer multi-frame integration transformer fuses the 8 frame embeddings into one video embedding, a CLIP text encoder embeds each class name, and a video-specific prompt generator conditions the text embeddings on the clip's patch features; the clip is scored against every name and the carried module softmaxes the scores over the names you supplied (196,585,729 parameters in all, trained fully supervised on Kinetics-400, published under the **MIT** licence). **The probabilities are a softmax over your label set**: a relative ranking that sums to 1, not a calibrated probability, and a set that omits the true class still yields a confident top-1.

What this notebook adds to inference is **adaptation of a closed label set on labelled clips**. The clips are 300 human-action clips of ten HMDB51 classes — brushing hair, doing a cartwheel, catching a ball, chewing, clapping hands, climbing, climbing stairs, diving into water, drawing a sword, dribbling a basketball — cut from films and web videos; the frozen model already ranks the right class first for **0.803** of the 66 held-out clips in the build record (chance is 0.100), so the honest question is narrow: does a bounded fine-tuning of the fusion head — the frame-integration transformer, the two visual projections and the prompt generator, 10,247,680 of the parameters — on 181 clips move the held-out **top-1 accuracy**, **top-3 accuracy**, **macro recall** and **macro F1** on a source-disjoint test split past the frozen model and two **non-adapted baselines**, and what does it do to the drawn clips the same head ranks? Nothing here is a claim about your videos or your classes: it is one seeded split of one small labelled set.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 9-file manifest with the tokenizer and processor files) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed. The pipeline runs in **float32 on every device**: the adapter is trained in float32 and overlays without a cast, and CPU, Tesla-class and consumer GPUs then run the same arithmetic.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled clip set, decode it into 8-frame clips, validate it and split it by source video without leakage; rank drawn clips through the public API and read the output contract correctly (a softmax over the supplied names, no calibrated probability, a `sample-sanity` report only when the true classes are known); measure the frozen model's held-out top-1 / top-3 accuracy, macro recall and F1 beside two non-adapted baselines; run a bounded fine-tuning of the fusion head with the cross-entropy over the closed label set, explicit hyperparameters and validation-based epoch selection; evaluate on a source-disjoint test split; look at the adapted rankings next to the frozen ones and the references, and at what the drawn clips do after the shared head was tuned; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** temporal localisation or per-frame labels (one ranking per clip), open-set or abstaining classification (the softmax always picks one of your names), video–text retrieval over a corpus, fine-tuning of the vision tower, the text tower, the text projection or the logit scale, evaluation on Kinetics-400 or the full HMDB51 protocol (only one seeded 300-clip sample of ten classes is scored here), non-English class names, and any claim that ten HMDB51 actions stand in for your videos. The repository exposes none of these.

**Who this notebook is for.** A learner who knows basic Python, has used Colab or Jupyter, and wants to see how a pretrained video–text model is adapted to a closed label set with a small labelled clip set, and how to read the result honestly — including what the pre-training already covered. No prior experience with video models or fine-tuning is assumed; each term is explained where it first matters and again in the **Glossary** at the end. CPU is adequate; a GPU is faster.

**Input → Model → Output.**

| | Ranking a clip | Bounded fine-tuning |
|---|---|---|
| Input | one clip of 8 frames and 2–32 free-text class names | labelled clips: 181 training and 53 validation HMDB51 clips in the sample, split by source video |
| Model | X-CLIP base/32: a CLIP frame encoder with cross-frame attention, a frame-integration transformer, a text encoder and a prompt generator | the fusion head (10.2 M parameters) trained with cross-entropy over the label set on cached tower features; validation top-1 chooses the epoch |
| Output | a ranking of the names with a softmax over them — relative, not calibrated | a safetensors adapter, and held-out top-1 / top-3, macro recall and macro F1 beside two baselines |

**How to use this notebook.** Choose a runtime (CPU works; a GPU is faster), then **Runtime → Run all**. Run all completes in one pass: Section 1 installs nothing into the notebook's own Python, so no restart is needed. Sections 1–3 are **infrastructure** — the isolated environment, the carried package and the model snapshot — and their cells are collapsed; you may run them without studying them. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited, and the defaults reproduce the recorded run. Before each principal result the notebook asks you to **Predict**; after it come **What to notice** and a collapsible **Check your reasoning** with a worked answer that names the run it quotes — the Kaggle T4 release run of 21 September 2026. Section 10 is a **change-one-thing experiment**, off by default. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Writing your predictions down is optional.

**Roadmap:** 1–3 infrastructure → 4 HMDB51 clips and a source-grouped split *(evaluation practice)* → 5 ranking drawn clips through the inference contract *(core concept: a softmax over your labels)* → 6 baselines and the pre-trained model, with its Kinetics-400 overlap *(evaluation practice)* → 7 fine-tuning the fusion head *(core concept)* → 8 held-out evaluation → 9 clips, drawings, export and reload *(engineering)* → 10 change one thing (optional) → conclude.

## Prerequisites

- **Learner:** basic Python and Colab or Jupyter familiarity; no prior experience with video models or fine-tuning. The notebook explains cross-frame attention, the frame-integration transformer, the prompt generator, the frozen-tower cache, top-k accuracy, macro recall and source grouping where they are first used; the Glossary repeats them.
- **Runtime:** a fresh supported **Linux x86_64** runtime (Google Colab, Kaggle or Linux Jupyter; CPU or CUDA). Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels (PyAV included), so the kernel's own Python version does not matter and nothing is installed into it. The default path uses CUDA automatically when present. The vision tower runs `EVAL_BATCH_SIZE` clips (8 × 8 frames) per forward and the build record measured 3 s to score 66 clips and 12 s for the eight epochs (caching the tower features for 181 + 53 clips took 6 s) on a Tesla T4, about 2 minutes of cell time for the whole path including the pinned install and the downloads; the build venv's CPU ran the same path in a few minutes. The pinned `torch==2.14.0` install, the 786 MB checkpoint and the 113 MB row group are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what a contrastive video–text model scores and why a softmax over a label set you chose is a ranking and not a probability; what top-1 accuracy, macro recall and macro F1 measure on a closed label set and why 66 clips give no dispersion; why clips cut from one source video must stay in one split.
- **Data contract:** records are `{id, frames, label}` — `frames` exactly `NUM_FRAMES` (8) PIL frames of one size with sides within 16..4,096 px (a longer clip is subsampled uniformly by `sample_frames`; a container file is decoded by `decode_clip`), `label` one of the closed label set (normalised like the candidate names: stripped, lower-cased, trailing full stop removed), and an optional `source` naming the video the clip was cut from. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 16..5,000 records, at least two labels and at least two clips per label; splitting de-duplicates by decoded pixels and keeps every clip of one source in one split. BYOD accepts one zip (or directory) of clips plus a `labels.csv` in the layout named above.
- **Validation is structural, not semantic:** every clip is decoded and every label checked against the set, but nothing checks that a label describes its clip — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads row group 0 of `default/test/0000.parquet` from `https://huggingface.co/datasets/mteb/HMDB51/resolve/<revision>/` at the immutable parquet-conversion revision `50bb2abb…` with HTTPS range requests (the parquet footer plus about 113 MB of row-group bytes out of a 481 MB shard), pinned by SHA-256 and byte total in the carried `samples.py` and refused on any mismatch. HMDB51 is published under the CC BY 4.0 licence (Serre Lab, Brown University; Kuehne et al. 2011); the `mteb/HMDB51` repository is a parquet repack of it; nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/xclip-base-patch32` snapshot (~790 MB in total) at revision `a2e27a78a2b5…`. No GitHub access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 49 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab and Kaggle import some packages, NumPy and often PyTorch among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env_<lock digest>/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 49 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins in the cell) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message.

The same cell then starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. It is the only cell that runs in the kernel (it is marked `# dimer: kernel cell`), and it is safe to re-run: a complete environment built from this exact lock is reused rather than rebuilt, and a live worker is kept with every variable created so far, so the cells after it keep working. To start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated environment and route later cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
    'av==18.1.0',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '254a3005495451bf42caf9fa2dabee9615f4e24fa29fac59a1b6ae6f19e5c1b4'
LOCKED_PACKAGES = 49
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
av==18.1.0 \
    --hash=sha256:025f84494cb23278498f03b0d8117d3e47a1cbc9c44b97eb31875cf02251e46b \
    --hash=sha256:08a9ae288299cfcbf739dba4ad0c53b9b71f45184303dd45947920d022fed695 \
    --hash=sha256:191224788d87af06c31784a395bb73f14b72f33d7f4871ace0157de2abdc6276 \
    --hash=sha256:3c8b1f8b46f99d52e2d8b0ed5d0cdadf172d24794d46e2077b16e44ed08e26ff \
    --hash=sha256:47bfc286e1bc9de7ab4681fc2b575cd2460a66919d31ffe1bd5aa54fae531a28 \
    --hash=sha256:5509ec12aaa19fd6601de13cfa6f4cdad450da07982118510592875d970454d6 \
    --hash=sha256:5548b79e2bf1f59b3e9aedc918a72d9dc45b9adaac10ff9470d5dbdda0002e47 \
    --hash=sha256:5cd9085028902c9880622bd37a12fd4b33060f06a52311f6f4867ca9f29a2c3b \
    --hash=sha256:6fc837cc51adf80331ac850779cd53b5d4c4460b0ebe9057a02a921c6736f19d \
    --hash=sha256:8a032e8d8ebc73dec079364b9b4a6837638a2d106e8472314e685ffbf163e700 \
    --hash=sha256:ab5ac081bc9eaf54109120d4e56284674fecfbe520d9aa1707c7fa911ec5f4d2 \
    --hash=sha256:ae75d8bb6467895ed1f8572ededf7ffa49eac07f6e483222f5d7d62a41d12f04 \
    --hash=sha256:b30a4e8d934558e19602b68998a4d9ac9f250fa0dacef216f7e8e40153b13316 \
    --hash=sha256:b36b0bae9e4c62f9487c99481ec15e4e3870fcc868522cd6d18fc2d6bfa04f01 \
    --hash=sha256:cf8a17466bef07765dbdecc9e66ed9b25d20b4e14f654fbf35345a58ac45fa0c \
    --hash=sha256:d49a5c542dfdc00f43c6cdb6cc41dac1781ee206fe180b56aa7433dfa816dfae \
    --hash=sha256:e4d48b9f12cad009cc72fe4f4099107de5e819c95f82767f4fd01a01481c0661 \
    --hash=sha256:e7ea063f6690193ea335a1d592d6e0274350d45e2ed6af83ee107cb90cbfd84f \
    --hash=sha256:ea1480b7a8d5405cb5f382b344731bf125fd2c1c6fae3964f6c48595628387ff
    # via xclip-video-classification-pipeline (pyproject.toml)
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.12 \
    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \
    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   xclip-video-classification-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   xclip-video-classification-pipeline (pyproject.toml)
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   xclip-video-classification-pipeline (pyproject.toml)
    #   torchvision
pyarrow==25.0.1 \
    --hash=sha256:0b1edbb2f385a6a65e9711b62ba86ac54a7816a3f8d17bb3e8a5929d65fb2485 \
    --hash=sha256:0b726ad7e7b669be982b0c71c07fe4b037d654354130da79a7902a669e93a66b \
    --hash=sha256:0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f \
    --hash=sha256:0fe7c8b6c03969b49c8c66182e4a18e3819ab92d07cfab5d8370c531b9369ef0 \
    --hash=sha256:119297a6dc197e45d9c6d4415f7814a67ffa36c180d26f68c154c58067ae782d \
    --hash=sha256:169d3429d5be7c752125890620f75a60776d38b0035eddae939651640822332e \
    --hash=sha256:25f8720bf6387d5dc2ebd2622112de630760419e4b66134405dd24110d15f37e \
    --hash=sha256:31e49a7888fcdf3a835da33ae777f6bb9a866334e5a789282fc26dcf426f7f15 \
    --hash=sha256:35935cd5de130aa5cf4dea052a63e6bf2e17006c35c3a468194242b9b2bf5956 \
    --hash=sha256:38a9a4b4b9613380e200641891495a56c3d5a98a092db4a870af9975e220471d \
    --hash=sha256:3f89685964f46e4216103c75483aac0c0692a5f72212d7ca835adba5ede56ce3 \
    --hash=sha256:4288f27577352d608ca08553b0865e4a9b3aa14820c5d95b53337218d609835b \
    --hash=sha256:4340f0ba6c1d2e13f21658de1d7c662ca2545018568d0030a1e9afca159d87e3 \
    --hash=sha256:44a9120ce5bd81936b8ab9a88076e3fd47c2c6838e0e43630fed83626aca81d9 \
    --hash=sha256:4facd65742a024a4a366328a1d2292062d72d6e023c1b7dda8d4c37544933a25 \
    --hash=sha256:51093dd9e10325fbdb3c10a2ae7c4806e5c822d94e74ae4938b26524a3323fee \
    --hash=sha256:514ddb60285631af068875550c90eddc181db3e8e63a032b1559be189e82f056 \
    --hash=sha256:5389cdf79447ed1515c9e31620e6e1e2302249564d603f2ad727d4f6d313e4c3 \
    --hash=sha256:59a2de54c0cbd954da861eee4d1d330f8e909c45b53455baef696380f2c55033 \
    --hash=sha256:60e89d8f13861a1f7f8d950fa54aebb8023b30734d0ac51ffa80beabe2df4bba \
    --hash=sha256:6109c94d8b9f3b17a041daca16cacb2f651ad8f1ef70a4232c2c0f37a23da2a8 \
    --hash=sha256:62cd0d785b8aa6675ee355f9fc02252a340f4441257c42674937826fd7594325 \
    --hash=sha256:6943e2fe7954d29d84de45d29d34c8dc36ce96570e67d89aa9976e650a4a9138 \
    --hash=sha256:6a1fdfc6659b6b19022f2e50627fb5cf7156a66c46bf4299379955cbe742382a \
    --hash=sha256:880523be3d29efcf83d3998835d206118ccf35e3871dbd2fb60408cf6b007a80 \
    --hash=sha256:8858d7bfc22e3f51529aeaa4077225029724623e4595dc9eff8c793935c34140 \
    --hash=sha256:9150a83248bfed9813ea3c3af74c3856c1984d444aa28e58bf7733b9750ddf6a \
    --hash=sha256:9171748cdf796972d85a4b60157c279913e242992e350c90c7450182a9838b2a \
    --hash=sha256:a4d6d5e9a3d1879a97c08ded0c797579b7965eafd0f0c26c30b45ccc06db939b \
    --hash=sha256:a4dd8bf99a8fac133efc0ed6a92f5fddbe2adba0d0f6dd720e39ba9855cea85c \
    --hash=sha256:aa0559502e1cd6254d6814614085dd9c5a3dd0419362978a936a3f68a9e5c3df \
    --hash=sha256:b7a296aac7a71fa0886c08e155ddb6c636a50013f801f6178daafa0f9e726188 \
    --hash=sha256:bddd0c4f7630c2a3ddf6347c1bdaa79d97bcf6bd445f9e60c816b7d77c85a5ae \
    --hash=sha256:bf0b672390cdcb640d7288f96b826d71ff4e9abb254a86c89890baf51a29cee6 \
    --hash=sha256:c7c534ec03c358a76ea3e505e74c1b6aef290af90c444dfd092dbfe23e755b85 \
    --hash=sha256:cab40b1edfef0262e0e5251aa2c58d75630f24d06dd7794480243acc001a1d7d \
    --hash=sha256:cc4aa407fde9fc660be3939e49ea31f50f3e9fec17c0ec63159f7711edd3efc9 \
    --hash=sha256:d51592cb7561e87877c506113e7adbf1342ab579e6c21f0ef44b8ba41cb74c80 \
    --hash=sha256:dda9470024204d7bbf2042b47c6e8a0e47a3eeb8e34405882dfaea6577e0c153 \
    --hash=sha256:df961f2e7ae9cf496459259d798652c70625f6c080650d6952f8c04053c58ee9 \
    --hash=sha256:eb6203482ff3746a5632303a7279ae0b5a304c46985b49ed1378cb350ea6728d \
    --hash=sha256:f3831aaa25c67a99f99dc8b05873cb9d64560390372e2aa197ce9dd4a3f06a44 \
    --hash=sha256:f729cfdbd36fd99d543b67a914d2de044c84ebe45be8b34902b299b608c15c8f
    # via xclip-video-classification-pipeline (pyproject.toml)
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   xclip-video-classification-pipeline (pyproject.toml)
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   xclip-video-classification-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via xclip-video-classification-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via xclip-video-classification-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via xclip-video-classification-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
# One environment per lock digest: a re-run of this cell, or a second Run all in the same runtime, reuses a complete
# environment built from this exact lock instead of rebuilding it; a different lock gets a different folder.
ISOLATED_ENV = Path(os.environ.get('DIMER_ISOLATED_ENV', 'dimer_isolated_env_' + LOCK_SHA256[:12])).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / 'bin' / 'python'
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + '_tools')
ISOLATED_READY = ISOLATED_ENV / '.dimer-lock-sha256'


def _isolated_environment_ready():
    return ISOLATED_PYTHON.is_file() and ISOLATED_READY.is_file() and ISOLATED_READY.read_text(encoding='utf-8').strip() == LOCK_SHA256


if SKIP_INSTALL:
    print('DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.')
elif _isolated_environment_ready():
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': True, 'lock_sha256': LOCK_SHA256[:16] + '...', 'locked_packages': LOCKED_PACKAGES, 'kernel_python': platform.python_version()})
else:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode('utf-8')).hexdigest() != LOCK_SHA256:
        raise RuntimeError('The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.')
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding='utf-8', newline='\n')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
        uv = ISOLATED_TOOLS / 'uv'
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ, MPLBACKEND='Agg')
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), 'venv', '--quiet', '--managed-python', '--python', MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), '-c', 'import platform; print(platform.python_version())'], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f'{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.')
    ISOLATED_READY.unlink(missing_ok=True)
    subprocess.run([str(uv), 'pip', 'install', '--quiet', '--python', str(ISOLATED_PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(lock_path)], env=uv_env, check=True)
    ISOLATED_READY.write_text(LOCK_SHA256 + '\n', encoding='utf-8')
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': False, 'isolated_python': isolated_version, 'kernel_python': platform.python_version(), 'locked_packages': LOCKED_PACKAGES, 'setup_seconds': round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        self.python = str(python)
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def alive(self):
        return self.proc.poll() is None

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

if SKIP_INSTALL:
    print('Routing disabled: the notebook runs in this kernel.')
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_route_to_isolated_runtime']
    # Idempotent: a live worker on this environment is kept, with every variable the later cells created, so
    # re-running this cell alone never strands the cells after it. A dead or different worker is replaced.
    _previous = globals().get('_DIMER_ISOLATED_RUNTIME')
    _reuse = _previous is not None and getattr(_previous, 'python', None) == str(ISOLATED_PYTHON) and _previous.proc.poll() is None
    if _reuse:
        _DIMER_ISOLATED_RUNTIME = _previous
    else:
        if _previous is not None and _previous.proc.poll() is None:
            _previous.close()
        _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print({'routed_to': str(ISOLATED_PYTHON), 'worker_pid': _DIMER_ISOLATED_RUNTIME.proc.pid, 'worker_reused': _reuse})

{'isolated_environment': '/content/dimer_isolated_env_254a30054954', 'reused': False, 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 49, 'setup_seconds': 54}
{'routed_to': '/content/dimer_isolated_env_254a30054954/bin/python', 'worker_pid': 1769, 'worker_reused': False}


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers`, `av` versions, and whether CUDA is available.

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import os
import platform
import sys

NOTEBOOK_SOURCE = {
    'repository': 'xclip-video-classification-pipeline',
    'repository_revision': '1a3e196925857475bb2266a048ad206f4c1d1953',
    'embedded_module': 'src/xclip_video_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/xclip_video_classification_pipeline/pipeline.py', 'src/xclip_video_classification_pipeline/metrics.py', 'src/xclip_video_classification_pipeline/samples.py'],
    'module_sha256': 'eae2c1c3a77ed5c73d34aa07782e98650415ca9738abf8a7382cfbcd9bccdcfb',
    'generator': 'build_notebook.py/2.2',
    'notebook_spec': '2.2',
}
import torch, transformers, av
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'executable': sys.executable, 'torch': torch.__version__, 'transformers': transformers.__version__, 'av': av.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'xclip-video-classification-pipeline', 'repository_revision': '1a3e196925857475bb2266a048ad206f4c1d1953', 'embedded_module': 'src/xclip_video_classification_pipeline/pipeline.py', 'embedded_modules': ['src/xclip_video_classification_pipeline/pipeline.py', 'src/xclip_video_classification_pipeline/metrics.py', 'src/xclip_video_classification_pipeline/samples.py'], 'module_sha256': 'eae2c1c3a77ed5c73d34aa07782e98650415ca9738abf8a7382cfbcd9bccdcfb', 'generator': 'build_notebook.py/2.2', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'executable': '/content/dimer_isolated_env_254a30054954/bin/python', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'av': '18.1.0', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/xclip_video_classification_pipeline/` @ `1a3e19692585`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/xclip_video_classification_pipeline/pipeline.py`

In [ ]:
"""Zero-shot video classification with the pinned ``microsoft/xclip-base-patch32`` checkpoint (X-CLIP), plus the
adaptation contract for a closed label set: corpus evaluation of labelled clips, bounded fine-tuning of the fusion
head (the frame-integration transformer, the two visual projections and the video-specific prompt generator) on
cached tower features, and a verified adapter artifact.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the X-CLIP architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed. A clip is a sequence of exactly
NUM_FRAMES PIL frames; the caller names the candidate classes as free text and receives a softmax over
those names — a relative ranking, not a calibrated probability.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageSequence

MODEL_ID = "microsoft/xclip-base-patch32"
MODEL_REVISION = "a2e27a78a2b5d802e894b8a1ef14f3a8ce490963"
MODEL_LICENSE = "mit"
MODEL_KEY = "xclip-base-patch32"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The checkpoint was trained on 8 frames per clip (config.json vision_config.num_frames); the temporal
# modules expect exactly that many, so a clip is exactly NUM_FRAMES frames and longer sequences are
# subsampled uniformly by sample_frames.
NUM_FRAMES = 8
# Frame preprocessing: shorter side resized to 224, centre crop 224x224, ImageNet mean/std
# (preprocessor_config.json), 32x32 patches -> 49 tokens per frame.
FRAME_SIZE = 224
# Input ceilings. Each label is one CLIP text query (77-token context); the label set is the caller's
# closed vocabulary for this request.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MIN_LABELS = 2
MAX_LABELS = 32
MAX_LABEL_CHARS = 64
MAX_TEXT_TOKENS = 77
WEIGHTS_FILE = "model.safetensors"

# Adaptation contract: the fusion head — the multi-frame integration transformer (`mit`), the two visual projections
# and the video-specific prompt generator — is the adapter; the vision tower (12 ViT-B/32 layers with cross-frame
# attention), the text tower, the text projection and the logit scale stay frozen, so their outputs are computed
# once per clip and label set and cached.
PARAMETER_COUNT = 196_585_729
HEAD_PARAMETERS = 10_247_680
FRAME_TOKENS = 50  # 1 CLS + 49 patch tokens per 224-px frame
VISION_WIDTH = 768
_TRAINABLE_PREFIXES = ("visual_projection.", "mit.", "prompts_visual_layernorm.", "prompts_visual_projection", "prompts_generator.")
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = 1
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 30  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
EVAL_BATCH_SIZE = 8  # clips per vision-tower forward (8 clips x NUM_FRAMES frames)
GRAD_CLIP = 1.0


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)


def _trainable_names(model: Any) -> list[str]:
    """The fusion head's tensors; the vision tower, the text tower, the text projection and the logit scale stay
    frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}"
        )
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):
        raise ValueError("artifact tensors must all belong to the fusion head (mit, visual projections, prompt generator)")
    adapter = manifest.get("adapter") or {}
    labels = adapter.get("labels")
    if not isinstance(labels, list) or len(labels) < MIN_LABELS:
        raise ValueError("artifact manifest must record adapter.labels, the closed label set the head was trained on")


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_labels(labels: Sequence[str]) -> list[str]:
    """Validate the candidate class names and normalise them: stripped, whitespace-collapsed,
    lower-cased, trailing full stop removed, distinct. Names are passed to the CLIP text tower as-is
    otherwise (the upstream zero-shot recipe uses plain class names such as "playing soccer")."""
    if isinstance(labels, str) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of class names, not a single string")
    if not MIN_LABELS <= len(labels) <= MAX_LABELS:
        raise ValueError(
            f"label count {len(labels)} outside MIN_LABELS {MIN_LABELS}..MAX_LABELS {MAX_LABELS}"
        )
    cleaned: list[str] = []
    for name in labels:
        if not isinstance(name, str):
            raise TypeError(f"label must be str, got {type(name).__name__}")
        text = " ".join(name.split()).strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("labels must not be empty")
        if len(text) > MAX_LABEL_CHARS:
            raise ValueError(
                f"label {text[:12]!r}... is {len(text)} chars > MAX_LABEL_CHARS {MAX_LABEL_CHARS}"
            )
        cleaned.append(text)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("labels must be distinct after normalisation")
    return cleaned


def validate_frame(frame: Any) -> Image.Image:
    if not isinstance(frame, Image.Image):
        raise TypeError(f"frame must be a PIL.Image.Image, got {type(frame).__name__}")
    width, height = frame.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"frame side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"frame side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return frame.convert("RGB")


def validate_clip(frames: Any) -> list[Image.Image]:
    """Exactly NUM_FRAMES PIL frames of one common size, each within the side ceilings."""
    if isinstance(frames, Image.Image) or not isinstance(frames, Sequence):
        raise TypeError("frames must be a sequence of PIL.Image.Image, not a single image")
    if len(frames) != NUM_FRAMES:
        raise ValueError(
            f"a clip is exactly NUM_FRAMES={NUM_FRAMES} frames, got {len(frames)}; "
            "use sample_frames to subsample a longer sequence"
        )
    rgb = [validate_frame(frame) for frame in frames]
    if len({frame.size for frame in rgb}) != 1:
        raise ValueError("all frames of a clip must have the same size")
    return rgb


def sample_frames(frames: Sequence[Image.Image], n: int = NUM_FRAMES) -> list[Image.Image]:
    """Pick ``n`` frames at evenly spaced indices (first and last included) from a longer sequence."""
    if isinstance(frames, Image.Image) or not isinstance(frames, Sequence):
        raise TypeError("frames must be a sequence of PIL.Image.Image")
    if len(frames) < n:
        raise ValueError(f"need at least {n} frames to sample {n}, got {len(frames)}")
    indices = np.linspace(0, len(frames) - 1, num=n).round().astype(int)
    return [frames[int(index)] for index in indices]


def frames_from_animation(image: Image.Image) -> list[Image.Image]:
    """Decode every frame of an animated image (GIF, WebP, APNG) that Pillow can open into RGB copies."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    frames = [frame.convert("RGB") for frame in ImageSequence.Iterator(image)]
    if not frames:
        raise ValueError("the image holds no frames")
    return frames


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        f"one clip of exactly NUM_FRAMES={NUM_FRAMES} PIL.Image.Image frames of one size (any mode, "
        "converted to RGB) plus MIN_LABELS..MAX_LABELS free-text class names"
    ),
    "frame_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "frames_per_clip": NUM_FRAMES,
    "labels": [MIN_LABELS, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "label_tokens": [1, MAX_TEXT_TOKENS],
    "preprocessing": (
        f"each frame: shorter side resized to {FRAME_SIZE}, centre crop {FRAME_SIZE}x{FRAME_SIZE}, ImageNet "
        "mean/std, 32x32 patches; labels normalised into one CLIP text query each (format_labels); the "
        "video embedding (frame features fused by the multi-frame integration transformer) is scored "
        "against each label embedding and the scores are softmaxed over the supplied labels"
    ),
    "output": (
        "one probability per supplied label (softmax over the label set: a relative ranking that sums to "
        "1 and is not calibrated), the raw logits, and the top-1 label"
    ),
}


def _check_inputs(frames: Any, labels: Any) -> tuple[list[Image.Image], list[str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``classify`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_clip(frames), format_labels(labels)


def validate_inputs(
    clips: Sequence[Sequence[Image.Image]],
    labels: Sequence[str],
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every clip is checked exactly as ``classify`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(clips, Image.Image) or not isinstance(clips, Sequence) or not clips:
        raise TypeError("clips must be a non-empty sequence of frame sequences")
    if clips and isinstance(clips[0], Image.Image):
        raise TypeError("clips must be a sequence of clips (each a sequence of frames), not one clip")
    if names is not None and len(names) != len(clips):
        raise ValueError(f"names has {len(names)} entries for {len(clips)} clips")
    checked_labels = format_labels(labels)
    observed = []
    for index, clip in enumerate(clips):
        rgb, _ = _check_inputs(clip, labels)
        observed.append(
            {
                "id": names[index] if names else f"clip-{index}",
                "n_frames": len(rgb),
                "frame_mode": clip[0].mode,
                "frame_size": list(rgb[0].size),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": observed,
        "labels": checked_labels,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    correct_labels: Sequence[str] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``correct_labels`` (one class name per result, in order; normalised like the labels) the report
    carries ``top1_accuracy`` over the clips, the chance baseline (mean of 1/n_labels) and one
    per-clip entry, verdict ``sample-sanity``; without them it is ``not-measurable`` and says what
    labelled data would make the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one classification result")
    base = {
        "task": "zero-shot video classification over a caller-supplied label set (top-1 over the labels)",
        "score_semantics": (
            "probabilities are a softmax over the supplied labels only: a relative ranking that sums to 1, "
            "not a calibrated probability, and a label set without the true class still yields a confident "
            "top-1"
        ),
        "sample_kind": sample_kind,
        "n_clips": len(results),
        "n_labels": [len(result["labels"]) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if correct_labels is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no correct labels were supplied for the classified clips",
            "needs": (
                "labelled clips from the deployment domain with a class vocabulary matching the labels "
                "(Kinetics-style annotations) scored with top-1/top-5 accuracy over many clips; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(correct_labels) != len(results):
        raise ValueError(f"correct_labels has {len(correct_labels)} entries for {len(results)} results")
    per_clip = []
    for result, correct in zip(results, correct_labels, strict=True):
        key = format_labels([correct, "__second__"])[0]
        if key not in result["labels"]:
            raise ValueError(f"correct label {correct!r} is not among the result's labels")
        per_clip.append(
            {
                "clip": result.get("clip"),
                "top1": result["top1"],
                "top1_probability": result["predictions"][0]["probability"],
                "correct_label": key,
                "correct": result["top1"] == key,
                "rank_of_correct": next(
                    index for index, entry in enumerate(result["predictions"]) if entry["label"] == key
                )
                + 1,
            }
        )
    chance = sum(1.0 / n for n in base["n_labels"]) / len(results)
    return {
        **base,
        "metrics": [
            {
                "id": "top1_accuracy",
                "value": sum(entry["correct"] for entry in per_clip) / len(per_clip),
                "estimation": f"{len(per_clip)} clip(s), no dispersion estimate",
            }
        ],
        "baselines": [{"id": "chance", "value": chance, "note": "mean of 1/n_labels over the clips"}],
        "per_clip": per_clip,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_clip)} clip(s) with caller-known classes; plumbing evidence, not a video "
            "classification benchmark"
        ),
        "needs": (
            "labelled clips from the deployment domain with a matching class vocabulary for any "
            "top-1/top-5 accuracy claim; Kinetics-400 and UCF101 are not bundled"
        ),
    }


@dataclass
class XClipVideoClassificationPipeline:
    """Zero-shot video classification over the pinned X-CLIP base/32 checkpoint."""

    _runner: Callable[[list[Image.Image], list[str]], np.ndarray]
    device: str
    _batch_runner: Callable[[list[list[Image.Image]], list[str]], np.ndarray] | None = None
    _model: Any = None
    _processor: Any = None
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None
    # Pinned-base values of every fusion-head tensor adapt() or load_artifact() has changed, kept the first time each
    # is about to change: every adaptation and every frozen evaluation after restore_base() reads the verified base,
    # never a previous run (review finding XCL-M2).
    _base_state: dict[str, Any] = field(default_factory=dict, repr=False)

    def _remember_base(self, names: Sequence[str]) -> None:
        model, _processor = self._require_model()
        state = model.state_dict()
        for name in names:
            if name not in self._base_state:
                self._base_state[name] = state[name].detach().clone()

    def restore_base(self) -> list[str]:
        """Put the pipeline back to the pinned base: copy the base values into every fusion-head tensor an earlier
        adapt() or load_artifact() changed and drop the adapter record, so `classify`, `evaluate` and a new adapt()
        read the untouched checkpoint. Returns the names of the tensors that differed from the base."""
        model, _processor = self._require_model()
        state = model.state_dict()
        # Only tensors whose live value differs from the base count as restored, so an adapt() on an unchanged head
        # reports "pinned base" without claiming an earlier run changed anything (t5-base 93a578f).
        restored = sorted(n for n, base in self._base_state.items() if not bool((state[n] == base).all()))
        if restored:
            model.load_state_dict(dict(self._base_state), strict=False)
            model.eval()
        self.adapter = None
        return restored

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> XClipVideoClassificationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import XCLIPModel, XCLIPProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        weight_sha256 = _weight_digest(root) if (root / MANIFEST_NAME).is_file() else None
        processor = XCLIPProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = XCLIPModel.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(frames: list[Image.Image], labels: list[str]) -> np.ndarray:
            # The frame list is handed to the snapshot's VideoMAE-style image processor directly: in
            # the pinned transformers release XCLIPProcessor's `videos=` keyword yields no
            # pixel_values (only `images=` does), and calling the two sub-processors makes the
            # contract explicit. One clip -> pixel_values (1, NUM_FRAMES, 3, 224, 224).
            pixel_values = processor.image_processor([frames], return_tensors="pt")["pixel_values"]
            text = processor.tokenizer(labels, padding=True, return_tensors="pt")
            with torch.inference_mode():
                outputs = model(
                    pixel_values=pixel_values.to(resolved_device),
                    input_ids=text["input_ids"].to(resolved_device),
                    attention_mask=text["attention_mask"].to(resolved_device),
                )
            return outputs.logits_per_video[0].float().cpu().numpy()

        def batch_runner(clips: list[list[Image.Image]], labels: list[str]) -> np.ndarray:
            pixel_values = processor.image_processor(clips, return_tensors="pt")["pixel_values"]
            text = processor.tokenizer(labels, padding=True, return_tensors="pt")
            with torch.inference_mode():
                outputs = model(
                    pixel_values=pixel_values.to(resolved_device),
                    input_ids=text["input_ids"].to(resolved_device),
                    attention_mask=text["attention_mask"].to(resolved_device),
                )
            return outputs.logits_per_video.float().cpu().numpy()

        return cls(runner, resolved_device, batch_runner, model, processor, weight_sha256, None)

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained")
        return self._model, self._processor

    def classify(self, frames: Sequence[Image.Image], labels: Sequence[str]) -> dict[str, Any]:
        """Rank ``labels`` for one clip of exactly NUM_FRAMES frames; probabilities are a softmax over them.

        The softmax is a relative ranking over the supplied labels, not a calibrated probability.
        """
        rgb, names = _check_inputs(frames, labels)
        logits = np.asarray(self._runner(rgb, names), dtype=np.float64).reshape(-1)
        if logits.shape != (len(names),) or not np.all(np.isfinite(logits)):
            raise RuntimeError(f"backend returned logits of shape {logits.shape} for {len(names)} labels")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs)
        predictions = [
            {"label": names[index], "probability": float(probs[index]), "logit": float(logits[index])}
            for index in order
        ]
        return {
            "predictions": predictions,
            "top1": predictions[0]["label"],
            "labels": names,
            "n_frames": len(rgb),
            "frame_size": list(rgb[0].size),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ------------------------------------------------------------------------------------------------------
    # Adaptation contract: batched classification, corpus evaluation, bounded fine-tuning, artifacts
    # ------------------------------------------------------------------------------------------------------

    @staticmethod
    def _result(logits: np.ndarray, names: list[str]) -> dict[str, Any]:
        logits = np.asarray(logits, dtype=np.float64).reshape(-1)
        if logits.shape != (len(names),) or not np.all(np.isfinite(logits)):
            raise RuntimeError(f"backend returned logits of shape {logits.shape} for {len(names)} labels")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs, kind="stable")
        predictions = [
            {"label": names[index], "probability": float(probs[index]), "logit": float(logits[index])}
            for index in order
        ]
        return {"predictions": predictions, "top1": predictions[0]["label"], "labels": names}

    def classify_batch(
        self,
        clips: Sequence[Sequence[Image.Image]],
        labels: Sequence[str],
        *,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> list[dict[str, Any]]:
        """Rank `labels` for many clips, `batch_size` clips per forward; one result (as `classify` returns it, minus
        the frame fields) per clip, in order. With an injected runner and no batch runner the clips are ranked one by
        one through the runner."""
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = format_labels(labels)
        checked = [validate_clip(clip) for clip in clips]
        out: list[dict[str, Any]] = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            if self._batch_runner is not None:
                logits = np.asarray(self._batch_runner(batch, names), dtype=np.float64)
                if logits.shape != (len(batch), len(names)):
                    raise RuntimeError(f"backend returned logits of shape {logits.shape} for {len(batch)} clips x {len(names)} labels")
                rows = list(logits)
            else:
                rows = [np.asarray(self._runner(clip, names), dtype=np.float64) for clip in batch]
            for clip, row in zip(batch, rows, strict=True):
                out.append({**self._result(row, names), "n_frames": len(clip), "frame_size": list(clip[0].size)})
            if progress is not None:
                progress(min(start + batch_size, len(checked)), len(checked))
        return out

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        labels: Sequence[str],
        *,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Rank the closed label set for every validated record and score the rankings against the record labels
        with ``metrics.classification_metrics`` (top-1 / top-3 accuracy, macro recall and F1, confusion). Works with
        an injected runner too."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, labels, min_records=1, max_records=MAX_EVAL_RECORDS, min_per_class=1)["records"]
        names = format_labels(labels)
        started = time.perf_counter()
        results = self.classify_batch([r["frames"] for r in checked], names, batch_size=batch_size, progress=progress)
        rankings = [[p["label"] for p in r["predictions"]] for r in results]
        metrics = classification_metrics(rankings, checked, names)
        return {
            **metrics,
            "labels": names,
            "predictions": [r["top1"] for r in results],
            "top1_probability": [r["predictions"][0]["probability"] for r in results],
            "rankings": rankings,
            "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def _encode_clips(self, records: Sequence[Mapping[str, Any]], batch_size: int, progress: Callable[[int, int], None] | None = None) -> tuple[Any, Any]:
        """Run the frozen vision tower once per clip: pooled CLS features (n, NUM_FRAMES, VISION_WIDTH) and patch
        features (n, NUM_FRAMES, FRAME_TOKENS - 1, VISION_WIDTH), kept on the model's device in float32."""
        model, processor = self._require_model()
        import torch

        cls_out, patch_out = [], []
        with torch.no_grad():  # not inference_mode: the cached tensors feed autograd during adapt
            for start in range(0, len(records), batch_size):
                batch = [r["frames"] for r in records[start : start + batch_size]]
                pixel_values = processor.image_processor(batch, return_tensors="pt")["pixel_values"].to(self.device)
                vision = model.vision_model(pixel_values=pixel_values.flatten(0, 1))
                cls_out.append(vision[1].reshape(len(batch), NUM_FRAMES, -1).clone())
                patch_out.append(vision[0][:, 1:, :].reshape(len(batch), NUM_FRAMES, FRAME_TOKENS - 1, -1).clone())
                if progress is not None:
                    progress(min(start + batch_size, len(records)), len(records))
        return torch.cat(cls_out), torch.cat(patch_out)

    def _encode_labels(self, names: Sequence[str]) -> Any:
        model, processor = self._require_model()
        import torch

        text = processor.tokenizer(list(names), padding=True, return_tensors="pt")
        with torch.no_grad():
            return model.get_text_features(input_ids=text["input_ids"].to(self.device), attention_mask=text["attention_mask"].to(self.device)).clone()

    def _head_logits(self, cls_features: Any, patch_features: Any, text_features: Any) -> Any:
        """The fusion head on cached tower features: exactly what `XCLIPModel.forward` computes after the towers
        (parity with the full forward is asserted by the model-backed tests)."""
        model, _ = self._require_model()
        import torch

        batch = cls_features.shape[0]
        video = model.visual_projection(cls_features.reshape(batch * NUM_FRAMES, -1)).view(batch, NUM_FRAMES, -1)
        video = model.mit(video)[1]
        img = model.prompts_visual_layernorm(patch_features.reshape(batch * NUM_FRAMES, FRAME_TOKENS - 1, -1))
        img = (img @ model.prompts_visual_projection).view(batch, NUM_FRAMES, -1, video.shape[-1]).mean(dim=1)
        text = text_features.unsqueeze(0).expand(batch, -1, -1)
        text = text + model.prompts_generator(text, img)
        video = video / video.norm(p=2, dim=-1, keepdim=True)
        text = text / text.norm(p=2, dim=-1, keepdim=True)
        return torch.einsum("bd,bkd->bk", video, text) * model.logit_scale.exp()

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None,
        labels: Sequence[str],
        *,
        epochs: int = 8,
        lr: float = 1e-5,
        batch_size: int = 16,
        seed: int = 0,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the fusion head on labelled clips with the cross-entropy over the closed label set
        (the model's own contrastive scoring, read as a classifier over `labels`). The frozen towers — the ViT-B/32
        vision encoder with its cross-frame attention and the CLIP text encoder — are run once per clip and label
        set under no gradient and their outputs are cached, so each step runs only the head; the logits equal the
        full model's exactly. AdamW (no weight decay), gradient clipping at `GRAD_CLIP`, seeded shuffling, no
        scheduler, no augmentation. Epoch 0 records the frozen model's validation metrics; the epoch with the highest
        validation top-1 accuracy (the earliest on ties) is kept. Every call starts from the pinned base: head tensors
        an earlier adapt() or load_artifact() changed are restored first, so epoch 0 is always the frozen model. On
        any exception the head and the adapter record this call found are put back."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
            raise ValueError("epochs must be an int in 1..100")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 128:
            raise ValueError("batch_size must be an int in 1..128")
        if not isinstance(lr, int | float) or isinstance(lr, bool) or not 0 < lr <= 1e-2:
            raise ValueError("lr must be a number in (0, 1e-2]")
        names = format_labels(labels)
        train_checked = validate_dataset(train, names)["records"]
        val_checked = validate_dataset(val, names, min_records=1, min_per_class=1)["records"] if val is not None else None
        model, _processor = self._require_model()
        import torch

        started = time.perf_counter()
        head_names = _trainable_names(model)
        params = {name: param for name, param in model.named_parameters() if name in set(head_names)}
        n_trainable = sum(p.numel() for p in params.values())
        # The head and adapter as this call found them: a failed call puts them back (the transactional contract),
        # while a successful one starts from the pinned base.
        backup = {name: param.detach().clone() for name, param in params.items()}
        previous_adapter = self.adapter
        restored = self.restore_base()
        self._remember_base(head_names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in params)
        cudnn_flags = torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
        try:
            model.eval()
            text_features = self._encode_labels(names)
            targets = torch.tensor([names.index(r["label"]) for r in train_checked], device=self.device)
            cls_train, patch_train = self._encode_clips(train_checked, EVAL_BATCH_SIZE)
            cached = None
            if val_checked is not None:
                cached = self._encode_clips(val_checked, EVAL_BATCH_SIZE)
            cache_seconds = round(time.perf_counter() - started, 3)

            def score_val() -> dict[str, Any] | None:
                if val_checked is None or cached is None:
                    return None
                with torch.no_grad():
                    rows = []
                    for start in range(0, len(val_checked), EVAL_BATCH_SIZE):
                        rows.append(self._head_logits(cached[0][start : start + EVAL_BATCH_SIZE], cached[1][start : start + EVAL_BATCH_SIZE], text_features))
                    logits = torch.cat(rows).float().cpu().numpy()
                rankings = [[names[i] for i in np.argsort(-row, kind="stable")] for row in logits]
                m = classification_metrics(rankings, val_checked, names)
                return {k: m[k] for k in ("top1_accuracy", "top3_accuracy", "macro_recall", "macro_f1", "n")}

            history: list[dict[str, Any]] = [{"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}]
            if progress is not None:
                progress(history[-1])
            best_epoch, best_acc = 0, (history[0]["val"] or {}).get("top1_accuracy", -1.0)
            best_state = {name: param.detach().clone() for name, param in params.items()}
            optimizer = torch.optim.AdamW(list(params.values()), lr=lr, weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            order = list(range(len(train_checked)))
            for epoch in range(1, epochs + 1):
                rng.shuffle(order)
                model.train()
                total, steps = 0.0, 0
                for start in range(0, len(order), batch_size):
                    idx = torch.tensor(order[start : start + batch_size], device=self.device)
                    optimizer.zero_grad(set_to_none=True)
                    logits = self._head_logits(cls_train[idx], patch_train[idx], text_features)
                    loss = torch.nn.functional.cross_entropy(logits, targets[idx])
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(list(params.values()), GRAD_CLIP)
                    optimizer.step()
                    total += float(loss.detach())
                    steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": round(total / max(steps, 1), 5), "val": score_val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                acc = (entry["val"] or {}).get("top1_accuracy")
                if val_checked is None or (acc is not None and acc > best_acc):
                    best_epoch, best_acc = epoch, acc if acc is not None else best_acc
                    best_state = {name: param.detach().clone() for name, param in params.items()}
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(best_state[name])
        except BaseException:
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(backup[name])
            model.eval()
            self.adapter = previous_adapter
            raise
        finally:
            for param in model.parameters():
                param.requires_grad_(False)
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "labels": names,
            "trainable_names": head_names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "highest validation top-1 accuracy" if val_checked is not None else "final epoch (no validation split)",
            "lr": lr,
            "seed": seed,
            "started_from": "pinned base"
            + (f" (restored {len(restored)} head tensors changed by an earlier run)" if restored else ""),
            "grad_clip": GRAD_CLIP,
            "objective": "cross-entropy over the closed label set on cached tower features",
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "cache_seconds": cache_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests, the label set and
        the training configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the fusion head."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        self.restore_base()
        self._remember_base(sorted(tensors))
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> XClipVideoClassificationPipeline:
        """Check the adapter manifest against the base snapshot's recorded weight digest, load the verified base, then
        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads
        a model."""
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/xclip_video_classification_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level measures for closed-set clip classification: top-1 and top-3 accuracy, macro recall and macro F1
over the label set, the per-label confusion, and two non-adapted baselines (chance, majority-training-label).

Every rate is computed over the supplied records with the supplied closed label set; nothing is calibrated and no
dispersion is estimated (one seeded split of one sample gives one number)."""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import format_labels` removed — names are kernel globals defined by the carried modules

TOP_K = 3

METRIC_DEFINITIONS = {
    "top1_accuracy": "fraction of clips whose highest-scoring label is the reference label",
    "top3_accuracy": f"fraction of clips whose reference label is among the {TOP_K} highest-scoring labels (1.0 by construction when the label set has {TOP_K} or fewer labels)",
    "macro_recall": "mean over the label set of the per-label recall (correct clips of that label / clips of that label); labels absent from the records are skipped",
    "macro_f1": "mean over the label set of the per-label F1 between precision and recall of the top-1 decision; labels absent from both references and predictions are skipped",
    "confusion": "reference label -> predicted top-1 label -> count",
    "baselines": "chance = 1 / |labels| top-1 by construction; majority = the most frequent training label predicted for every clip",
}


def _rank(ranking: Sequence[str], reference: str) -> int:
    for index, label in enumerate(ranking):
        if label == reference:
            return index + 1
    return len(ranking) + 1


def classification_metrics(
    rankings: Sequence[Sequence[str]], records: Sequence[Mapping[str, Any]], labels: Sequence[str]
) -> dict[str, Any]:
    """Score one ranking of `labels` (best first) per record against the record's `label`."""
    label_set = list(format_labels(labels))
    if len(rankings) != len(records):
        raise ValueError(f"{len(rankings)} rankings for {len(records)} records")
    if not records:
        raise ValueError("no records to score")
    refs = [format_labels([r["label"], "__other__"])[0] for r in records]
    for ref in refs:
        if ref not in label_set:
            raise ValueError(f"record label {ref!r} is not in the label set")
    ranks = [_rank(list(ranking), ref) for ranking, ref in zip(rankings, refs, strict=True)]
    tops = [list(ranking)[0] if ranking else "" for ranking in rankings]
    confusion: dict[str, dict[str, int]] = {label: {} for label in label_set}
    for ref, top in zip(refs, tops, strict=True):
        confusion[ref][top] = confusion[ref].get(top, 0) + 1
    support = Counter(refs)
    predicted = Counter(tops)
    recalls, f1s = [], []
    per_label = {}
    for label in label_set:
        tp = confusion[label].get(label, 0)
        n_ref, n_pred = support.get(label, 0), predicted.get(label, 0)
        recall = tp / n_ref if n_ref else None
        precision = tp / n_pred if n_pred else None
        f1 = None
        if n_ref or n_pred:
            f1 = (2 * tp / (n_ref + n_pred)) if (n_ref + n_pred) else None
        per_label[label] = {"support": n_ref, "predicted": n_pred, "correct": tp, "recall": recall, "precision": precision, "f1": f1}
        if recall is not None:
            recalls.append(recall)
        if f1 is not None:
            f1s.append(f1)
    k = min(TOP_K, len(label_set))
    return {
        "n": len(records),
        "n_labels": len(label_set),
        "top1_accuracy": sum(r == 1 for r in ranks) / len(ranks),
        "top3_accuracy": sum(r <= k for r in ranks) / len(ranks),
        "macro_recall": sum(recalls) / len(recalls) if recalls else 0.0,
        "macro_f1": sum(f1s) / len(f1s) if f1s else 0.0,
        "mean_rank": sum(ranks) / len(ranks),
        "per_label": per_label,
        "confusion": confusion,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def chance_baseline(records: Sequence[Mapping[str, Any]], labels: Sequence[str]) -> dict[str, Any]:
    """Uniform guessing over the label set: top-1 = 1 / |labels|, top-3 = min(3, |labels|) / |labels|."""
    label_set = list(format_labels(labels))
    n = len(label_set)
    return {
        "n": len(records),
        "n_labels": n,
        "top1_accuracy": 1.0 / n,
        "top3_accuracy": min(TOP_K, n) / n,
        "macro_recall": 1.0 / n,
        "macro_f1": 1.0 / n,
        "kind": "chance",
    }


def majority_label(train: Sequence[Mapping[str, Any]], labels: Sequence[str]) -> str:
    """The most frequent training label (ties broken by label-set order)."""
    label_set = list(format_labels(labels))
    counts = Counter(format_labels([r["label"], "__other__"])[0] for r in train)
    return max(label_set, key=lambda label: (counts.get(label, 0), -label_set.index(label)))


def majority_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], labels: Sequence[str]
) -> dict[str, Any]:
    """Predict the most frequent training label for every clip, ranking the rest in label-set order."""
    label_set = list(format_labels(labels))
    top = majority_label(train, label_set)
    ranking = [top] + [label for label in label_set if label != top]
    out = classification_metrics([ranking] * len(records), records, label_set)
    out["kind"] = "majority"
    out["label"] = top
    return out

**Module 3/3:** `src/xclip_video_classification_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled clip datasets for the adaptation contract: the digest-pinned HMDB51 sample, the record contract and its
structural validation, source-disjoint splitting, and the BYOD loader.

A record is ``{id, frames, label}`` where ``frames`` is a list of exactly ``NUM_FRAMES`` PIL frames of one size
(uniformly sampled from a clip; sides within the pipeline's ceilings) and ``label`` the class name the clip is asked
to be ranked under — one of the caller's closed label set, normalised like ``format_labels`` normalises the
candidate names. An optional ``source`` names the video the clip was cut from; splits are made disjoint on it.

The default sample is drawn from HMDB51 (Kuehne et al. 2011; Serre Lab, Brown University; **CC BY 4.0**) as
repacked into parquet on the Hugging Face Hub (``mteb/HMDB51``) at an immutable revision: the first row group of
the test shard is read with one HTTPS range request (about 113 MB; the shard's declared size is checked first and
the row group's content is refused unless its SHA-256 matches the pin). It holds every test clip of the first ten
action classes (30 each, 300 clips) plus 18 clips of the eleventh, which are dropped so every class is complete.
Clips are MPEG-4 AVI files of 76..647 frames at 30 fps, mostly 320×240, decoded with PyAV into ``NUM_FRAMES``
uniformly spaced frames. Several clips are cut from one source video (its name is the prefix of the file name), so
the sample is split by source, never by clip.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import random
import re
import urllib.request
import zipfile
from collections import defaultdict
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, NUM_FRAMES, format_labels, sample_frames, validate_clip` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "HMDB51 (test shard), first parquet row group: the ten first action classes"
CORPUS_REPO = "mteb/HMDB51"
CORPUS_REVISION = "50bb2abb741074ef0868e59ba51af710c825cd1f"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "default/test/0000.parquet"
CORPUS_BYTES = 480_591_821
CORPUS_ROWS = 1_530
CORPUS_ROW_GROUPS = 1  # of 5; 318 clips
CORPUS_LICENSE = "CC BY 4.0 (HMDB51, Serre Lab, Brown University; Kuehne, Jhuang, Garrote, Poggio, Serre, ICCV 2011; parquet repack mteb/HMDB51)"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
# SHA-256 over the concatenated AVI bytes + UTF-8 label index of each row, in row order, and that byte total.
ROW_GROUP_PINS: dict[int, tuple[str, int]] = {
    0: ("c1eb3e9cd30abc8d3deadb1e82b982a57d2baf2035bfdf99d1343dcdb1c09bc1", 112_858_960),
}
DEFAULT_CACHE_DIR = Path("weights") / "hmdb51"

# The HMDB51 class vocabulary (parquet `label` feature names) and the ten classes the sample keeps, with the
# natural-language label the CLIP text tower is asked to rank (the upstream zero-shot recipe uses plain phrases).
HMDB51_CLASSES = ("brush_hair", "cartwheel", "catch", "chew", "clap", "climb", "climb_stairs", "dive", "draw_sword", "dribble", "drink", "eat", "fall_floor", "fencing", "flic_flac", "golf", "handstand", "hit", "hug", "jump", "kick", "kick_ball", "kiss", "laugh", "pick", "pour", "pullup", "punch", "push", "pushup", "ride_bike", "ride_horse", "run", "shake_hands", "shoot_ball", "shoot_bow", "shoot_gun", "sit", "situp", "smile", "smoke", "somersault", "stand", "swing_baseball", "sword", "sword_exercise", "talk", "throw", "turn", "walk", "wave")
SAMPLE_CLASS_TEXT: dict[str, str] = {
    "brush_hair": "brushing hair",
    "cartwheel": "doing a cartwheel",
    "catch": "catching a ball",
    "chew": "chewing",
    "clap": "clapping hands",
    "climb": "climbing",
    "climb_stairs": "climbing stairs",
    "dive": "diving into water",
    "draw_sword": "drawing a sword",
    "dribble": "dribbling a basketball",
}
SAMPLE_LABELS: tuple[str, ...] = tuple(SAMPLE_CLASS_TEXT.values())
SAMPLE_CLIPS_PER_CLASS = 30

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"test": 6, "validation": 4}  # clips per class the source-grouped split reserves at least; the rest train
SAMPLE_DIGEST = "ddf8c1730d64ff80b5ec691522f09461220c6817dee70b55197cfd12b9191f4e"  # dataset_digest over the three default splits together; tests pin it
MIN_RECORDS = 16
MAX_RECORDS = 5_000
MIN_PER_CLASS = 2
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_CLIP_SUFFIXES = (".avi", ".mp4", ".mov", ".mkv", ".webm", ".gif", ".webp")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a
    parquet footer and one row group without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "xclip-video-classification-pipeline"})
        with urllib.request.urlopen(request, timeout=600) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _declared_size(url: str) -> int:
    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "xclip-video-classification-pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)
        length = response.headers.get("Content-Length")
    if length is None:
        raise ValueError(f"{url}: no Content-Length in the HEAD response")
    return int(length)


def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    for row in rows:
        data = row["video"]["bytes"]
        label = str(row["label"]).encode("utf-8")
        digest.update(data)
        digest.update(label)
        total += len(data) + len(label)
    return digest.hexdigest(), total


def fetch_corpus(
    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None
) -> dict[int, list[dict[str, Any]]]:
    """Return the pinned row groups as lists of `{video, path, label}` (AVI bytes, file name, class index), from the
    cache (one parquet file per row group) or the Hub (footer + the row group, over range requests). Every row group's
    content is refused unless its SHA-256 and byte total match `ROW_GROUP_PINS`; a fresh fetch also checks the shard's
    declared size and row count."""
    import pyarrow.parquet as pq

    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)
    out: dict[int, list[dict[str, Any]]] = {}
    reader = None
    for group in wanted:
        if group not in ROW_GROUP_PINS:
            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")
        local = cache / f"test-rg{group}.parquet"
        rows: list[dict[str, Any]] | None = None
        if local.is_file():
            rows = pq.read_table(local).to_pylist()
            if _group_digest(rows) != ROW_GROUP_PINS[group]:
                rows = None  # stale or corrupt cache: refetch
        if rows is None:
            if reader is None:
                if opener is not None:
                    reader = pq.ParquetFile(opener(CORPUS_URL))
                else:
                    declared = _declared_size(CORPUS_URL)
                    if declared != CORPUS_BYTES:
                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")
                    reader = pq.ParquetFile(io.BufferedReader(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES), buffer_size=1 << 20))
                if reader.metadata.num_rows != CORPUS_ROWS:
                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")
            table = reader.read_row_group(group, columns=["video", "label"])
            rows = table.to_pylist()
            digest, total = _group_digest(rows)
            if (digest, total) != ROW_GROUP_PINS[group]:
                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")
            pq.write_table(table, local)
        out[group] = [{"video": r["video"]["bytes"], "path": str(r["video"]["path"] or ""), "label": int(r["label"])} for r in rows]
    return out


def decode_clip(data: bytes, n: int = NUM_FRAMES) -> list[Image.Image]:
    """Decode a video file held in memory with PyAV and return `n` uniformly spaced RGB frames (first and last
    included). Every frame is decoded (the container's declared frame count is not trusted); only the chosen ones
    are kept."""
    import av

    frames: list[Image.Image] = []
    with av.open(io.BytesIO(data)) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        for frame in container.decode(stream):
            frames.append(frame.to_image())
    if len(frames) < n:
        raise ValueError(f"clip holds {len(frames)} frames, fewer than the {n} a clip needs")
    return sample_frames(frames, n)


def source_of(path: str, class_name: str) -> str:
    """The source an HMDB51 clip was cut from, keyed per action: `<source video>/<class>` from the file name
    `<source>_<class>_<tags>_<n>.avi`. Clips cut from one source video for one action share its scene and actor, so
    a split is made disjoint on this key (one film can still contribute different actions to different splits)."""
    match = re.match(rf"^(.*)_{re.escape(class_name)}_(.*)_(\d+)\.avi$", path)
    return f"{match.group(1) if match else Path(path).stem}/{class_name}"


def read_corpus(groups: Mapping[int, Sequence[Mapping[str, Any]]]) -> list[dict[str, Any]]:
    """Decode the verified row groups into records (one per clip of the ten sample classes; the partial eleventh
    class is dropped)."""
    out = []
    for group in sorted(groups):
        for index, row in enumerate(groups[group]):
            class_name = HMDB51_CLASSES[row["label"]]
            if class_name not in SAMPLE_CLASS_TEXT:
                continue
            out.append(
                {
                    "id": f"hmdb51-test-{group}-{index}",
                    "frames": decode_clip(row["video"]),
                    "label": SAMPLE_CLASS_TEXT[class_name],
                    "class": class_name,
                    "source": source_of(row["path"], class_name),
                    "source_row_group": group,
                }
            )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded source-grouped draw, per class: whole source videos go to the test split until it holds at least
    `sizes['test']` clips of that class, then to validation until `sizes['validation']`, and the rest train. Sources
    are taken smallest first (seeded order among equals), so the held-out splits stay near their targets and the
    largest sources train."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_class: dict[str, dict[str, list[dict[str, Any]]]] = defaultdict(lambda: defaultdict(list))
    for record in records:
        by_class[record["label"]][record.get("source") or record["id"]].append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for label in sorted(by_class):
        sources = sorted(by_class[label])
        rng.shuffle(sources)
        sources.sort(key=lambda source: len(by_class[label][source]))  # stable: seeded order among equal sizes
        counts = {"test": 0, "validation": 0, "train": 0}
        for source in sources:
            clips = by_class[label][source]
            if counts["test"] < sizes["test"]:
                out["test"].extend(clips)
                counts["test"] += len(clips)
            elif counts["validation"] < sizes["validation"]:
                out["validation"].extend(clips)
                counts["validation"] += len(clips)
            else:
                out["train"].extend(clips)
                counts["train"] += len(clips)
        if counts["train"] < MIN_PER_CLASS or counts["test"] < sizes["test"]:
            raise ValueError(f"class {label!r} has too few sources to reserve {sizes} clips and keep a training source")
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _check_record(record: Any, index: int, labels: Sequence[str] | None) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/frames/label")
    for key in ("id", "frames", "label"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        frames = validate_clip(record["frames"])
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where}: {exc}") from exc
    if not isinstance(record["label"], str) or not record["label"].strip():
        raise ValueError(f"{where}: label must be a non-empty str")
    label = format_labels([record["label"], "__other__"])[0]
    if labels is not None and label not in labels:
        raise ValueError(f"{where}: label {label!r} is not in the label set {list(labels)}")
    item = {"id": rid, "frames": frames, "label": label}
    for key in ("class", "source", "source_row_group"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    labels: Sequence[str] | None = None,
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    min_per_class: int = MIN_PER_CLASS,
) -> dict[str, Any]:
    """Structural validation of a labelled-clip dataset; raises ValueError before any model import. With `labels`
    every record's label must belong to that closed set; without, the set is the labels the records carry."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, frames, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    label_set = list(format_labels(labels)) if labels is not None else None
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index, label_set)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    counts: dict[str, int] = defaultdict(int)
    for item in checked:
        counts[item["label"]] += 1
    if label_set is None:
        label_set = sorted(counts)
        if len(label_set) < 2:
            raise ValueError("a labelled-clip dataset needs at least two distinct labels")
    thin = [label for label, n in counts.items() if n < min_per_class]
    if thin:
        raise ValueError(f"every label needs at least {min_per_class} clips; too few for {sorted(thin)}")
    widths = [r["frames"][0].width for r in checked]
    heights = [r["frames"][0].height for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "labels": list(label_set),
        "per_label": {label: counts.get(label, 0) for label in label_set},
        "n_sources": len({r.get("source") or r["id"] for r in checked}),
        "frames_per_clip": NUM_FRAMES,
        "frame_width": {"min": min(widths), "max": max(widths)},
        "frame_height": {"min": min(heights), "max": max(heights)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def clip_digest(frames: Sequence[Image.Image]) -> str:
    """SHA-256 of the decoded RGB pixels of every frame (size-prefixed) — the identity a split is made disjoint on."""
    digest = hashlib.sha256()
    for frame in frames:
        rgb = frame.convert("RGB")
        digest.update(f"{rgb.width}x{rgb.height}:".encode())
        digest.update(rgb.tobytes())
    return digest.hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, clip digest, label)."""
    parts = sorted(f"{r['id']}:{clip_digest(r['frames'])}:{r['label']}" for r in records)
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no clip (by decoded-pixel digest) and no source video appears in two splits (leakage check)."""
    seen_clip: dict[str, str] = {}
    seen_source: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = clip_digest(record["frames"])
            if key in seen_clip and seen_clip[key] != name:
                raise ValueError(f"clip {record['id']!r} appears in both {seen_clip[key]} and {name}")
            seen_clip[key] = name
            source = record.get("source")
            if source:
                if source in seen_source and seen_source[source] != name:
                    raise ValueError(f"source video {source!r} has clips in both {seen_source[source]} and {name}")
                seen_source[source] = name
    return {name: len(records) for name, records in splits.items()}


def _split_counts(per_label: int, val_fraction: float, test_fraction: float) -> tuple[int, int, int]:
    """(train, validation, test) clips that `split_dataset` aims to cut from one label holding `per_label` clips."""
    n_test = max(1, round(per_label * test_fraction))
    n_val = round(per_label * val_fraction)
    return per_label - n_test - n_val, n_val, n_test


def min_byod_records(n_labels: int = 2, *, val_fraction: float = 0.15, test_fraction: float = 0.2) -> dict[str, int]:
    """The smallest balanced BYOD set (one source per clip) `split_dataset` accepts with `n_labels` labels: every
    label needs `MIN_PER_CLASS` training clips, a test clip and (when `val_fraction` > 0) a validation clip, and the
    training split needs `MIN_RECORDS` clips in total. With the default fractions two labels need 24 clips (12 per
    label, split 8 / 2 / 2), three need 27 (9 per label), and four need 24 (6 per label)."""
    if isinstance(n_labels, bool) or not isinstance(n_labels, int) or n_labels < 2:
        raise ValueError("n_labels must be an int >= 2")
    for per_label in range(1, MAX_RECORDS + 1):
        train, val, test = _split_counts(per_label, val_fraction, test_fraction)
        if train >= MIN_PER_CLASS and (val >= 1 or val_fraction == 0) and train * n_labels >= MIN_RECORDS:
            return {"per_label": per_label, "total": per_label * n_labels, "train_per_label": train,
                    "validation_per_label": val, "test_per_label": test}
    raise ValueError("no dataset size satisfies these fractions")


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded, label-stratified shuffle of a BYOD dataset into train/validation/test after de-duplicating clips.
    Records that carry a `source` keep every clip of one source in one split, **across labels** (a film with two
    actions never straddles splits): whole sources are dealt in a seeded order to the test split while every label
    they carry still needs test clips, then to validation, and the rest train."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = clip_digest(record["frames"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    rng = random.Random(seed)
    by_source: dict[str, list[dict[str, Any]]] = defaultdict(list)
    per_label: dict[str, int] = defaultdict(int)
    for record in unique:
        by_source[record.get("source") or record["id"]].append(record)
        per_label[record["label"]] += 1
    target = {"test": {}, "validation": {}}
    for label, n in per_label.items():
        _train, target["validation"][label], target["test"][label] = _split_counts(n, val_fraction, test_fraction)
    have: dict[str, dict[str, int]] = {"test": defaultdict(int), "validation": defaultdict(int)}
    out: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    sources = sorted(by_source)
    rng.shuffle(sources)
    for source in sources:
        clips = by_source[source]
        labels = {r["label"] for r in clips}
        part = "train"
        for name in ("test", "validation"):
            if all(have[name][label] < target[name][label] for label in labels):
                part = name
                break
        out[part].extend(clips)
        if part != "train":
            for record in clips:
                have[part][record["label"]] += 1
    need = min_byod_records(max(len(per_label), 2), val_fraction=val_fraction, test_fraction=test_fraction)
    advice = (
        f"with {len(per_label)} labels and these split fractions supply at least {need['total']} distinct clips, "
        f"{need['per_label']} per label from separate sources"
    )
    for name in ("train", "test", "validation") if val_fraction > 0 else ("train", "test"):
        present = {r["label"] for r in out[name]}
        starved = sorted(label for label in per_label if label not in present)
        if starved:
            raise ValueError(f"the {name} split holds no clip of {starved[:5]}; {advice}")
    if len(out["train"]) < MIN_RECORDS:
        raise ValueError(
            f"the train split holds {len(out['train'])} clips; at least {MIN_RECORDS} are required — {advice}"
        )
    for part in out.values():
        rng.shuffle(part)
    return out


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding video clips (any container PyAV or Pillow decodes: AVI, MP4, MOV,
    MKV, WebM, GIF, WebP) and a `labels.csv` with the columns `file` and `label` (and optionally `id` and `source`);
    every clip file must have a label row and every row a clip. Each clip is decoded into NUM_FRAMES uniform frames."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file() and not any(part == "__MACOSX" or part.startswith(".") for part in file.parts):
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                parts = Path(info.filename).parts
                if not info.is_dir() and not any(part == "__MACOSX" or part.startswith(".") for part in parts):
                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "labels.csv" not in members:
        raise ValueError("BYOD data must include labels.csv with the columns file and label")
    rows = list(csv.DictReader(io.StringIO(members["labels.csv"].decode("utf-8-sig"))))
    if not rows or "file" not in rows[0] or "label" not in rows[0]:
        raise ValueError("labels.csv must have the columns file and label")
    out = []
    for line, row in enumerate(rows, start=2):  # line 1 is the header
        name = Path(str(row.get("file", "")).strip()).name
        where = f"labels.csv line {line} (file {name!r})"
        if name not in members:
            raise ValueError(f"{where}: names a missing clip (not in the dataset)")
        try:
            if name.lower().endswith((".gif", ".webp")):
                pass  # standalone rewrite (build_notebook.py): `from .pipeline import frames_from_animation` removed — names are kernel globals defined by the carried modules

                frames = sample_frames(frames_from_animation(Image.open(io.BytesIO(members[name]))))
            else:
                frames = decode_clip(members[name])
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"{where}: not a decodable clip of at least {NUM_FRAMES} frames") from exc
        rid = str(row.get("id", "") or "").strip()
        item = {"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "frames": frames, "label": str(row.get("label", ""))}
        if str(row.get("source", "") or "").strip():
            item["source"] = str(row["source"]).strip()
        out.append(item)
    listed = {Path(str(r.get("file", "")).strip()).name for r in rows}
    unlisted = [n for n in members if n != "labels.csv" and n.lower().endswith(_CLIP_SUFFIXES) and n not in listed]
    if unlisted:
        raise ValueError(f"{len(unlisted)} clip file(s) have no labels.csv row, e.g. {unlisted[0]}")
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, frame size, label, class, source, provenance) in the BYOD `labels.csv` column layout plus
    extras (`file` names the id; the frames themselves are not written)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "file", "width", "height", "frames", "label", "class", "source", "source_row_group"])
        for r in records:
            writer.writerow([r["id"], f"{r['id']}.avi", r["frames"][0].width, r["frames"][0].height, len(r["frames"]), r["label"], r.get("class", ""), r.get("source", ""), r.get("source_row_group", "")])
    return out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `9`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a2e27a78a2b5…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `XClipVideoClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot, then load the model
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "xclip-base-patch32",
  "modelId": "microsoft/xclip-base-patch32",
  "revision": "a2e27a78a2b5d802e894b8a1ef14f3a8ce490963",
  "files": [
    {
      "path": "README.md",
      "bytes": 2728,
      "sha256": "5b4c29da30c14c41dadc2320cedb68d7bc30147aa171c04ce8d3d7b8acf8da91"
    },
    {
      "path": "config.json",
      "bytes": 4718,
      "sha256": "13bb919d1ef16f3b80b03cdf5c575d688dfafd21d12566849cc557b00e058ce3"
    },
    {
      "path": "merges.txt",
      "bytes": 524657,
      "sha256": "f526393189112391ce6f9795d4695f704121ce452c3aad1f5335cc41337eba85"
    },
    {
      "path": "model.safetensors",
      "bytes": 786414772,
      "sha256": "abf286e8cdd0612761c3e42d3a55eca998382dfa67a04a0f3fdcdfa4f150cdbb"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 309,
      "sha256": "c14b2b5c8f26a754df62235ba79d1ca63cfdd9b3de76ee688e4a30ea1e5c6986"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 472,
      "sha256": "c4864a9376a8401918425bed71fc14fc0e81f9b59ec45c1cf96cccb2df508eac"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2224041,
      "sha256": "a75dc79c6ec004a7e2d346c20e0af8d29aa2b251ea356964718aef8b8f052e80"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 965,
      "sha256": "7e28810d55aafb02f6b216f05fa1208a4aac62abf794796288a70314eed5ddf3"
    },
    {
      "path": "vocab.json",
      "bytes": 862328,
      "sha256": "5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363"
    }
  ],
  "totalBytes": 790034990
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = XClipVideoClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'microsoft/xclip-base-patch32', 'revision': 'a2e27a78a2b5d802e894b8a1ef14f3a8ce490963', 'license': 'mit', 'files': 9, 'total_bytes': 790034990}


{'weights_dir': '/content/weights/xclip-base-patch32', 'fetched': ['README.md', 'config.json', 'merges.txt', 'model.safetensors', 'preprocessor_config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']}


{'verified_files': 9, 'revision': 'a2e27a78a2b5d802e894b8a1ef14f3a8ce490963'}


{'device': 'cuda:0', 'source': 'local-snapshot'}


## 4. HMDB51 clips, the labels and the source-grouped split

`fetch_corpus` returns the pinned row group from the cache under `weights/hmdb51/` or the Hub at the pinned parquet-conversion revision — `pyarrow` reads the shard's footer and exactly that row group over HTTPS range requests; the cached file is re-hashed and a fetched row group refused on any SHA-256 or byte-total mismatch — and `read_corpus` turns each row of the ten sample classes into a record: the AVI decoded by PyAV into 8 uniformly spaced frames (mostly 320 × 240), the class rendered as the natural-language label the text tower is asked to rank, and the source video the clip was cut from. `build_sample_dataset` draws a seeded **source-grouped** split: per class, whole source videos go to the test split until it holds six clips, then to validation until four, and the rest train (181 / 53 / 66 in the build record). `validate_dataset` then checks every record against the contract and the label set, `check_split_disjoint` asserts no clip (by decoded-pixel digest) and no source video is shared, and the training split's summary table is written to `outputs/xclip_video_classification_train.csv`.

Look for: 300 clips of ten labels, three digests, and four refusal probes — a duplicate id, a label outside the set, a clip with three frames, and a dataset too small to use — each rejected before the model does anything.

*Evaluation practice.* **Bring your own data (optional):** set `USE_BYOD = True` and either `BYOD_PATH` (a zip or a folder holding `labels.csv` and the clips, as a path in this runtime — this works on Colab, Kaggle and Jupyter) or leave `BYOD_PATH` empty to upload exactly one zip through the Colab dialog; then choose **Run after** from this cell. This cell first puts the model back to the pinned base, so Section 6 scores the pre-trained head — also after a `SPLIT_SEED` change, when clips the adapted head trained on could land in the test split. The effective minimum is 24 clips for two labels.

**Predict before running:** several HMDB51 clips are cut from one film. What would a split by clip measure?

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
# A re-run after Section 7 (BYOD, a new split or new phrasing): Section 6 and Section 7's epoch 0 must read the pinned base.
had_adapter = pipe.adapter is not None
restored_tensors = pipe.restore_base()
if had_adapter or restored_tensors:
    print({'restored_pinned_base': len(restored_tensors), 'note': 'the fine-tuned fusion head was put back to the checkpoint; Sections 5-7 start from it again'})
if USE_BYOD:
    if BYOD_PATH.strip():
        byod_zip = Path(BYOD_PATH.strip()).expanduser()
        if not byod_zip.exists():
            raise FileNotFoundError(f'BYOD_PATH {BYOD_PATH!r} does not exist (relative paths start at {Path.cwd()}): give a .zip or a folder holding labels.csv and the clips.')
        file_name = byod_zip.name
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('USE_BYOD is True but BYOD_PATH is empty, and the upload dialog exists only in Google Colab: on Kaggle or Jupyter put the zip (or folder) in the runtime and set BYOD_PATH to its path.') from None
        uploaded = files.upload() or {}
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one .zip file (received {len(uploaded)}; a cancelled dialog sends none): run this cell again.')
        file_name, payload = next(iter(uploaded.items()))
        if not file_name.lower().endswith('.zip'):
            raise ValueError(f'{file_name}: upload one .zip holding labels.csv and the clips.')
        byod_zip = Path('work') / 'byod.zip'
        byod_zip.parent.mkdir(parents=True, exist_ok=True)
        byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    LABELS = validate_dataset(records)['labels']
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records), 'labels': len(LABELS), 'effective_minimum': min_byod_records(len(LABELS))['total']}
    if len(splits['test']) < 5 * len(LABELS):
        print({'caution': f"only {len(splits['test'])} held-out test clips for {len(LABELS)} labels (fewer than 5 per label): rates move in large steps; add clips before reading them"})
else:
    t0 = time.perf_counter()
    corpus_groups = fetch_corpus(cache_dir='weights/hmdb51')
    fetch_seconds = round(time.perf_counter() - t0, 1)
    corpus = read_corpus(corpus_groups)
    LABELS = list(SAMPLE_LABELS)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'row_groups': len(corpus_groups), 'rows': sum(len(v) for v in corpus_groups.values()), 'clips_kept': len(corpus), 'bytes': sum(len(r['video']) for v in corpus_groups.values() for r in v), 'fetch_seconds': fetch_seconds, 'decode_seconds': round(time.perf_counter() - t0 - fetch_seconds, 1)}
dataset_manifests = {name: validate_dataset(part, LABELS, min_records=1, min_per_class=1) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
validate_dataset(train_records, LABELS)  # the training split must satisfy the full record bounds
write_dataset_csv(train_records, 'outputs/xclip_video_classification_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'labels': LABELS})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'per_label': manifest['per_label'], 'sources': manifest['n_sources'], 'width': manifest['frame_width'], 'height': manifest['frame_height'], 'digest': manifest['digest'][:16] + '...'}})


def frame_strip(frames, height=120):
    tiles = [f.resize((round(f.width * height / f.height), height)) for f in frames]
    strip = Image.new('RGB', (sum(t.width for t in tiles) + 4 * (len(tiles) - 1), height), (255, 255, 255))
    x = 0
    for tile in tiles:
        strip.paste(tile, (x, 0))
        x += tile.width + 4
    return strip


example = train_records[0]
frame_strip(example['frames']).save('outputs/xclip_video_classification_example_clip.png')
print({'example': {'id': example['id'], 'frames': len(example['frames']), 'frame_size': list(example['frames'][0].size), 'label': example['label'], 'source': example.get('source')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:16]],
    'label outside the set': [{**train_records[0], 'label': 'juggling'}, *train_records[1:16]],
    'clip with three frames': [{**train_records[0], 'frames': train_records[0]['frames'][:3]}, *train_records[1:16]],
    'too small': train_records[:8],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe, LABELS)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

{'data_source': 'HMDB51 (test shard), first parquet row group: the ten first action classes @ 50bb2abb7410 (CC BY 4.0 (HMDB51, Serre Lab, Brown University; Kuehne, Jhuang, Garrote, Poggio, Serre, ICCV 2011; parquet repack mteb/HMDB51))', 'raw_rows': {'row_groups': 1, 'rows': 318, 'clips_kept': 300, 'bytes': 112858624, 'fetch_seconds': 11.4, 'decode_seconds': 29.6}, 'splits': {'train': 181, 'validation': 53, 'test': 66}, 'labels': ['brushing hair', 'doing a cartwheel', 'catching a ball', 'chewing', 'clapping hands', 'climbing', 'climbing stairs', 'diving into water', 'drawing a sword', 'dribbling a basketball']}
{'train': {'n': 181, 'per_label': {'brushing hair': 18, 'doing a cartwheel': 20, 'catching a ball': 18, 'chewing': 18, 'clapping hands': 20, 'climbing': 16, 'climbing stairs': 19, 'diving into water': 19, 'drawing a sword': 18, 'dribbling a basketball': 15}, 'sources': 42, 'width': {'min': 288, 'max': 480}, 'height': {'min': 240, 'max': 240}, 'digest': '297eeabaf4f4ed29...'}}
{'

**What to notice:** 300 clips of ten labels, 181 / 53 / 66, `sources` per split, the three digests and the four refusals.

<details><summary>Check your reasoning</summary>How well the model recognises a film it has already seen: clips cut from one source share its scene, lighting and actor. The sample's split keeps a source together *per action*, so one film can still contribute different actions to different splits; BYOD's `split_dataset` keeps a source together across labels.</details>

## 5. Rank five drawn clips through the inference contract

*Core concept.* The probabilities are a softmax over the names you supplied: they always sum to one, so a set that omits the true class still gets a confident top-1. **Predict before running:** on simple cartoon clips, will the human-action model rank the right motion first?

The inference contract is exercised as the inference-only tutorial exercised it: five deterministic 8-frame clips at 320 × 240 drawn with Pillow — a ball rolling right, a ball bouncing, a square growing, a sun setting, a ball standing still — with the five class names that describe them; a different clip family from the human-action videos, and clips the adapted model will be asked to rank again in Section 9. `validate_inputs` applies exactly the checks `classify` applies (exactly `NUM_FRAMES` frames of one size within the side ceilings, 2..`MAX_LABELS` distinct names of at most `MAX_LABEL_CHARS` characters) and one combined manifest records the request; a three-frame clip is validated too and its rejection recorded as a finding. `classify` returns one probability per supplied name — **the probabilities are a softmax over your label set**, and the label set is a **caller-owned request parameter**. The per-request `evaluation_report` with the intended classes is `sample-sanity`: `top1_accuracy` against the chance baseline on five cartoons is plumbing evidence, not a video-classification benchmark — video-classification accuracy needs labelled clips of the deployment domain, which Section 6 supplies. The inference-only card recorded 1/5 (chance 1/5): the model was trained on human-action video and does not read the motion of flat drawn shapes, which the notebook keeps as a finding rather than tuning the drawings until they pass.

In [ ]:
def synthetic_clip(kind, n=8, size=(320, 240)):
    """An 8-frame cartoon clip drawn with Pillow (no text): sky, green ground and one moving element."""
    frames = []
    for index in range(n):
        t = index / (n - 1)
        sky = (135, 206, 235)
        if kind == 'the sun setting':
            sky = (int(135 * (1 - t) + 30 * t), int(206 * (1 - t) + 40 * t), int(235 * (1 - t) + 80 * t))
        frame = Image.new('RGB', size, sky)
        d = ImageDraw.Draw(frame)
        d.rectangle([0, 170, 320, 240], fill=(60, 179, 75))  # ground
        if kind == 'a ball rolling to the right':
            x = 30 + t * 230
            d.ellipse([x, 130, x + 40, 170], fill=(220, 40, 40))
        elif kind == 'a ball bouncing up and down':
            y = 130 - abs(np.sin(t * np.pi * 2)) * 100
            d.ellipse([140, y, 180, y + 40], fill=(220, 40, 40))
        elif kind == 'a square growing larger':
            s = 10 + t * 90
            d.rectangle([160 - s / 2, 120 - s / 2, 160 + s / 2, 120 + s / 2], fill=(40, 70, 200))
        elif kind == 'the sun setting':
            y = 30 + t * 140
            d.ellipse([240, y, 290, y + 50], fill=(255, 215, 0))
        elif kind == 'a ball standing still':
            d.ellipse([140, 130, 180, 170], fill=(220, 40, 40))
        frames.append(frame)
    return frames


DRAWN_LABELS = ['a ball rolling to the right', 'a ball bouncing up and down', 'a square growing larger', 'the sun setting', 'a ball standing still']
drawn_clips = [synthetic_clip(kind) for kind in DRAWN_LABELS]
drawn_names = [f"synthetic_{kind.replace(' ', '_')}_8x320x240" for kind in DRAWN_LABELS]
drawn_digests = {name: hashlib.sha256(b''.join(np.asarray(frame.convert('RGB')).tobytes() for frame in clip)).hexdigest() for name, clip in zip(drawn_names, drawn_clips, strict=True)}
print({'ceilings': {'NUM_FRAMES': NUM_FRAMES, 'FRAME_SIZE': FRAME_SIZE, 'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_LABELS': MIN_LABELS, 'MAX_LABELS': MAX_LABELS, 'MAX_LABEL_CHARS': MAX_LABEL_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'MIN_PER_CLASS': MIN_PER_CLASS, 'EVAL_BATCH_SIZE': EVAL_BATCH_SIZE, 'device': pipe.device}})
input_manifest = validate_inputs(drawn_clips, DRAWN_LABELS, names=drawn_names)
try:
    validate_inputs([drawn_clips[0][:3]], DRAWN_LABELS)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'three-frame-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/xclip_video_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'drawn_clips': len(drawn_clips), 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def rank_drawings(pipeline, label):
    results, timings = [], []
    for name, clip in zip(drawn_names, drawn_clips, strict=True):
        started = time.perf_counter()
        result = pipeline.classify(clip, DRAWN_LABELS)
        timings.append(round(time.perf_counter() - started, 3))
        results.append({**result, 'clip': name, 'rgb_sha256': drawn_digests[name]})
    checks = {
        'probabilities_sum_to_one': all(abs(sum(p['probability'] for p in r['predictions']) - 1.0) < 1e-6 for r in results),
        'one_entry_per_label': all([p['label'] for p in r['predictions']] and sorted(p['label'] for p in r['predictions']) == sorted(r['labels']) for r in results),
        'ranked_descending': all(all(a['probability'] >= b['probability'] for a, b in zip(r['predictions'], r['predictions'][1:], strict=False)) for r in results),
        'identity_reported': all(r['model_id'] == MODEL_ID and r['model_revision'] == MODEL_REVISION for r in results),
    }
    if not all(checks.values()):
        raise RuntimeError(f'classify output failed a sanity check: {checks}')
    report = evaluation_report(results, DRAWN_LABELS, sample_kind='synthetic (drawn in this notebook)')
    with open(f'outputs/xclip_video_classification_drawing_{label}.json', 'w', encoding='utf-8') as handle:
        json.dump({'results': results, 'report': report}, handle, indent=2, ensure_ascii=False)
    print({label: {'seconds': timings, 'checks': checks, 'top1': [(r['clip'].split('_8x')[0], r['top1'], round(r['predictions'][0]['probability'], 3)) for r in results], 'verdict': report['verdict'], 'top1_accuracy': report['metrics'][0]['value'], 'chance': report['baselines'][0]['value']}})
    return results, timings, checks, report


frozen_drawn, frozen_drawn_timings, frozen_drawn_checks, frozen_drawn_report = rank_drawings(pipe, 'frozen')

{'ceilings': {'NUM_FRAMES': 8, 'FRAME_SIZE': 224, 'MIN_IMAGE_SIDE': 16, 'MAX_IMAGE_SIDE': 4096, 'MIN_LABELS': 2, 'MAX_LABELS': 32, 'MAX_LABEL_CHARS': 64, 'MAX_TEXT_TOKENS': 77, 'MIN_RECORDS': 16, 'MAX_RECORDS': 5000, 'MIN_PER_CLASS': 2, 'EVAL_BATCH_SIZE': 8, 'device': 'cuda:0'}}
{'drawn_clips': 5, 'manifest_verdict': 'accepted', 'findings': 1}


{'frozen': {'seconds': [0.863, 0.083, 0.078, 0.078, 0.073], 'checks': {'probabilities_sum_to_one': True, 'one_entry_per_label': True, 'ranked_descending': True, 'identity_reported': True}, 'top1': [('synthetic_a_ball_rolling_to_the_right', 'a ball standing still', 0.689), ('synthetic_a_ball_bouncing_up_and_down', 'a ball standing still', 0.512), ('synthetic_a_square_growing_larger', 'a ball standing still', 0.427), ('synthetic_the_sun_setting', 'a ball standing still', 0.459), ('synthetic_a_ball_standing_still', 'a ball standing still', 0.567)], 'verdict': 'sample-sanity', 'top1_accuracy': 0.2, 'chance': 0.2}}


**What to notice:** each drawn clip's top-1 and its probability, and the drawn-label top-1 rate.

<details><summary>Check your reasoning</summary>No. In the Kaggle T4 release run the pre-trained model ranked `a ball standing still` first for all five drawn clips (probabilities 0.43–0.69), a drawn-label top-1 of 0.20. A softmax over five names you chose will always name one of them, confidently; the drawings are plumbing evidence, not a measurement.</details>

## 6. Baselines and the frozen model on the test clips

Two non-adapted baselines frame the adaptation, each scored by `classification_metrics` (carried in `metrics.py`): **top-1 accuracy** (the highest-scoring name is the reference label), **top-3 accuracy** (the reference is among the three highest), **macro recall** (the mean per-label recall, so a class the model never predicts counts fully) and **macro F1**, with the per-label confusion beside them. The **chance** baseline guesses uniformly: top-1 = 1 / 10 by construction. The **majority-label** baseline predicts the most frequent training label for every clip: what the label distribution buys without looking at the frames. The **frozen model** is scored by `pipe.evaluate`, which ranks the closed label set for every clip in batches of `EVAL_BATCH_SIZE` and returns the rankings with the rates. Call it **the pre-trained model scored on our label phrasing** rather than zero-shot: X-CLIP was trained fully supervised on Kinetics-400, and most of these ten actions are in that training vocabulary. Four labels have an exact or near-exact Kinetics-400 class (`brushing hair`, `doing a cartwheel`, `clapping hands`, `dribbling a basketball`), four have related classes (catching or throwing a ball, several kinds of climbing and diving, sword fighting), and only `chewing` and `climbing stairs` have none. HMDB51 and Kinetics both draw on web video, so overlap at the video level cannot be ruled out either. **What to look for:** the per-label recall against that overlap.

**Predict before running:** which labels will the pre-trained model find hardest — those with a Kinetics-400 counterpart or those without?

In [ ]:
METRICS = ('top1_accuracy', 'top3_accuracy', 'macro_recall', 'macro_f1')

baseline_chance = chance_baseline(test_records, LABELS)
baseline_majority = majority_baseline(train_records, test_records, LABELS)
print({'chance_baseline': {k: round(baseline_chance[k], 3) for k in METRICS}, 'n': baseline_chance['n'], 'n_labels': baseline_chance['n_labels']})
print({'majority_baseline': {k: round(baseline_majority[k], 3) for k in METRICS}, 'label': baseline_majority['label']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, LABELS, batch_size=EVAL_BATCH_SIZE)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'mean_rank': round(frozen_test['mean_rank'], 2), 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'per_label_recall': {label: round(row['recall'], 2) for label, row in frozen_test['per_label'].items()}})
print({'definitions': frozen_test['definitions']})
for record, ranking, probability in zip(test_records[:4], frozen_test['rankings'][:4], frozen_test['top1_probability'][:4], strict=True):
    print({'id': record['id'], 'reference': record['label'], 'frozen_top3': ranking[:3], 'top1_probability': round(probability, 3)})
frozen_verdict = 'pre-trained model above both baselines' if frozen_test['top1_accuracy'] > max(baseline_chance['top1_accuracy'], baseline_majority['top1_accuracy']) else 'a baseline matches or beats the pre-trained model'
print({'frozen_vs_baselines': frozen_verdict})

{'chance_baseline': {'top1_accuracy': 0.1, 'top3_accuracy': 0.3, 'macro_recall': 0.1, 'macro_f1': 0.1}, 'n': 66, 'n_labels': 10}
{'majority_baseline': {'top1_accuracy': 0.091, 'top3_accuracy': 0.288, 'macro_recall': 0.1, 'macro_f1': 0.017}, 'label': 'doing a cartwheel'}


{'frozen_model_test': {'top1_accuracy': 0.803, 'top3_accuracy': 0.939, 'macro_recall': 0.802, 'macro_f1': 0.792}, 'n': 66, 'mean_rank': 1.42, 'verdict': 'measured', 'seconds': 3.6}
{'per_label_recall': {'brushing hair': 1.0, 'doing a cartwheel': 1.0, 'catching a ball': 1.0, 'chewing': 0.43, 'clapping hands': 0.5, 'climbing': 1.0, 'climbing stairs': 1.0, 'diving into water': 0.67, 'drawing a sword': 0.71, 'dribbling a basketball': 0.71}}
{'definitions': {'top1_accuracy': 'fraction of clips whose highest-scoring label is the reference label', 'top3_accuracy': 'fraction of clips whose reference label is among the 3 highest-scoring labels (1.0 by construction when the label set has 3 or fewer labels)', 'macro_recall': 'mean over the label set of the per-label recall (correct clips of that label / clips of that label); labels absent from the records are skipped', 'macro_f1': 'mean over the label set of the per-label F1 between precision and recall of the top-1 decision; labels absent from b

**What to notice:** the two baselines, the frozen top-1 / top-3 / macro F1, and the per-label recall.

<details><summary>Check your reasoning</summary>In the Kaggle T4 release run (21 September 2026) the pre-trained model ranked the right label first for 0.803 of the 66 test clips (top-3 0.939, macro F1 0.792), against chance at 0.100 and the majority label at 0.091 — below chance because the training split's most frequent label is under-represented in the test split. Its weakest labels were `chewing` (0.43), which has no Kinetics-400 counterpart, `clapping hands` (0.50), `diving into water` (0.67) and `drawing a sword` (0.71). Much of the 0.803 is recognition of classes the model was trained on, not transfer.</details>

## 7. Bounded fine-tuning of the fusion head

`pipe.adapt` trains only the fusion head — the multi-frame integration transformer, the visual projection, the prompt-side visual layer norm and projection, and the video-specific prompt generator: 10,247,680 of 196,585,729 parameters — while the ViT-B/32 vision tower with its cross-frame attention, the CLIP text tower, the text projection and the logit scale stay frozen. The loss is the **cross-entropy over the closed label set**: the model's own contrastive scores, read as a classifier over the ten names. Because both towers are frozen, their outputs — the pooled CLS and the 49 patch features of every frame, and the ten label embeddings — are computed once under no gradient and cached (the **frozen-tower cache**), and each step runs only the head on those cached features: the logits equal the full model's exactly, at a fraction of the cost. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates; every epoch is scored on the 53 validation clips, and the epoch with the **highest validation top-1 accuracy** (the earliest on ties) is kept.

Watch the validation top-1 rise from 0.811 to 0.962 (epoch 2 in the build record) while the loss drops from about 0.44 to 0.00: the adapted head ranks 60 of 66 held-out clips first (frozen 53), and the reference is in the top three for 65. The learning rate is deliberately small — a head this size memorises 181 clips within an epoch or two at 1e-4, and the validation curve is then flat from the first epoch.

*Core concept.* Every call to `pipe.adapt` starts from the **pinned base**: head tensors an earlier call (or an artifact) changed are put back first, so epoch 0 is always the pre-trained model and re-running Sections 7–8 with a changed field repeats the comparison validly. To compare a change side by side without replacing the default exports, use Section 10.

**Predict before running:** will validation keep the last epoch, or an early one?

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


settings = {'epochs': EPOCHS, 'lr': LEARNING_RATE, 'batch_size': BATCH_SIZE}
if settings != {'epochs': 8, 'lr': 1e-5, 'batch_size': 16}:
    print({'note': 'changed settings: this run starts again from the pinned base and replaces the default results of Sections 8-9; Section 10 compares a change side by side instead', 'settings': settings})
t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, LABELS, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'labels': adapt_result['labels'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'objective': adapt_result['objective'], 'cache_seconds': adapt_result['cache_seconds'], 'started_from': adapt_result['started_from'], 'seconds': adapt_seconds})

{'epoch': 0, 'train_loss': None, 'val_top1_accuracy': 0.811, 'val_top3_accuracy': 0.925, 'val_macro_recall': 0.803, 'val_macro_f1': 0.767, 'note': 'frozen model'}


{'epoch': 1, 'train_loss': 0.4367, 'val_top1_accuracy': 0.925, 'val_top3_accuracy': 0.962, 'val_macro_recall': 0.92, 'val_macro_f1': 0.916}


{'epoch': 2, 'train_loss': 0.0819, 'val_top1_accuracy': 0.962, 'val_top3_accuracy': 0.981, 'val_macro_recall': 0.96, 'val_macro_f1': 0.955}
{'epoch': 3, 'train_loss': 0.0101, 'val_top1_accuracy': 0.962, 'val_top3_accuracy': 0.981, 'val_macro_recall': 0.96, 'val_macro_f1': 0.956}


{'epoch': 4, 'train_loss': 0.0015, 'val_top1_accuracy': 0.962, 'val_top3_accuracy': 0.981, 'val_macro_recall': 0.96, 'val_macro_f1': 0.956}
{'epoch': 5, 'train_loss': 0.0005, 'val_top1_accuracy': 0.962, 'val_top3_accuracy': 0.981, 'val_macro_recall': 0.96, 'val_macro_f1': 0.956}


{'epoch': 6, 'train_loss': 0.0003, 'val_top1_accuracy': 0.962, 'val_top3_accuracy': 0.981, 'val_macro_recall': 0.96, 'val_macro_f1': 0.956}
{'epoch': 7, 'train_loss': 0.0003, 'val_top1_accuracy': 0.962, 'val_top3_accuracy': 0.981, 'val_macro_recall': 0.96, 'val_macro_f1': 0.956}


{'epoch': 8, 'train_loss': 0.0002, 'val_top1_accuracy': 0.962, 'val_top3_accuracy': 0.981, 'val_macro_recall': 0.96, 'val_macro_f1': 0.956}
{'labels': ['brushing hair', 'doing a cartwheel', 'catching a ball', 'chewing', 'clapping hands', 'climbing', 'climbing stairs', 'diving into water', 'drawing a sword', 'dribbling a basketball'], 'trainable_parameters': 10247680, 'total_parameters': 196585729, 'best_epoch': 2, 'selection': 'highest validation top-1 accuracy', 'objective': 'cross-entropy over the closed label set on cached tower features', 'cache_seconds': 7.261, 'started_from': 'pinned base', 'seconds': 14.6}


**What to notice:** the validation top-1 per epoch, the training loss, and `best_epoch`.

<details><summary>Check your reasoning</summary>An early one: in the release run validation top-1 rose from 0.811 (epoch 0, the pre-trained model) to 0.962 and epoch 2 was kept, while the training loss fell towards zero — the head was memorising 181 clips. The earliest best epoch wins ties, so later epochs that only fit the training clips better are not chosen.</details>

## 8. Held-out evaluation

The test clips were never used for training or epoch selection; no clip appears in two splits, and no source video appears in two splits *for the same action* (the sample's split key is per action). The adapted model is scored exactly as the frozen model was in Section 6 and the four systems are put side by side. Read it in this order: **top-1 accuracy** first (the measure the epoch was selected on — the build record measured 0.803 → **0.909**), then **macro F1** (0.792 → 0.905), then **top-3 accuracy** (0.939 → 0.985), then the per-label recall to see which classes moved (`chewing` 0.43 → 0.86, `diving into water` 0.67 → 1.00, `drawing a sword` 0.71 → 1.00; down: `dribbling a basketball` 0.71 → 0.57). The cell records verdicts — `improved`, `no gain` or `worse` against the frozen model, and whether the adapted model beats both baselines — instead of asserting them: the epoch is chosen on validation, so the test split can still move the other way, and export, reload and the result still run. Sixty-six clips from one seeded split give **no dispersion estimate** — one clip is 1.5 points — so the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on ten HMDB51 actions says nothing about other actions, other cameras or your videos until you measure them.

**Predict before running:** which labels will fine-tuning help most — those the pre-trained model already handled, or those it missed?

In [ ]:
adapted_test = pipe.evaluate(test_records, LABELS, batch_size=EVAL_BATCH_SIZE)
adapted_val = pipe.evaluate(val_records, LABELS, batch_size=EVAL_BATCH_SIZE)
comparison = {metric: {'chance': round(baseline_chance[metric], 3), 'majority': round(baseline_majority[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['per_label_recall'] = {label: {'frozen': round(frozen_test['per_label'][label]['recall'], 2), 'adapted': round(adapted_test['per_label'][label]['recall'], 2), 'support': adapted_test['per_label'][label]['support']} for label in LABELS}
def direction(new, old):
    return 'improved' if new > old else ('no gain' if new == old else 'worse')
# Reported verdicts, not assertions: a fine-tune that does not help on the test split is a result to record.
comparison['verdicts'] = {'frozen_vs_baselines': frozen_verdict, **{f'adapted_vs_frozen_{metric}': direction(adapted_test[metric], frozen_test[metric]) for metric in METRICS}, 'adapted_above_both_baselines': bool(adapted_test['top1_accuracy'] > max(baseline_chance['top1_accuracy'], baseline_majority['top1_accuracy']))}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'labels': LABELS,
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'chance': baseline_chance, 'majority': {k: v for k, v in baseline_majority.items() if k not in ('definitions',)}},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'definitions'},
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'definitions'},
    'test_metrics': adapted_test,
    'per_clip': [{'id': r['id'], 'reference': r['label'], 'source': r.get('source'), 'frozen_top1': f, 'frozen_ranking': fr, 'adapted_top1': a, 'adapted_ranking': ar} for r, f, fr, a, ar in zip(test_records, frozen_test['predictions'], frozen_test['rankings'], adapted_test['predictions'], adapted_test['rankings'], strict=True)],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/xclip_video_classification_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'verdicts': comparison['verdicts']})
print({'report': 'outputs/xclip_video_classification_evaluation_report.json', 'adapted_beats_both_baselines': adapted_test['top1_accuracy'] > max(baseline_chance['top1_accuracy'], baseline_majority['top1_accuracy'])})

{'top1_accuracy': {'chance': 0.1, 'majority': 0.091, 'frozen': 0.803, 'adapted': 0.909}}
{'top3_accuracy': {'chance': 0.3, 'majority': 0.288, 'frozen': 0.939, 'adapted': 0.985}}
{'macro_recall': {'chance': 0.1, 'majority': 0.1, 'frozen': 0.802, 'adapted': 0.91}}
{'macro_f1': {'chance': 0.1, 'majority': 0.017, 'frozen': 0.792, 'adapted': 0.905}}
{'delta_vs_frozen': {'top1_accuracy': 0.106, 'top3_accuracy': 0.045, 'macro_recall': 0.107, 'macro_f1': 0.112}}
{'per_label_recall': {'brushing hair': {'frozen': 1.0, 'adapted': 1.0, 'support': 6}, 'doing a cartwheel': {'frozen': 1.0, 'adapted': 1.0, 'support': 6}, 'catching a ball': {'frozen': 1.0, 'adapted': 1.0, 'support': 7}, 'chewing': {'frozen': 0.43, 'adapted': 0.86, 'support': 7}, 'clapping hands': {'frozen': 0.5, 'adapted': 0.67, 'support': 6}, 'climbing': {'frozen': 1.0, 'adapted': 1.0, 'support': 8}, 'climbing stairs': {'frozen': 1.0, 'adapted': 1.0, 'support': 6}, 'diving into water': {'frozen': 0.67, 'adapted': 1.0, 'support': 6}, '

**What to notice:** `delta_vs_frozen`, the per-label recall rows and the `verdicts`.

<details><summary>Check your reasoning</summary>Mostly the ones it missed. In the release run top-1 rose from 0.803 to 0.909 (macro F1 0.792 → 0.905, top-3 0.939 → 0.985). The gain came from four labels — `chewing` 0.43 → 0.86, `diving into water` 0.67 → 1.00, `drawing a sword` 0.71 → 1.00, `clapping hands` 0.50 → 0.67 — while `dribbling a basketball` lost one clip (0.71 → 0.57). On 66 clips one clip is 1.5 points, and `chewing`, the label with no Kinetics-400 counterpart, is where tuning had most to add.</details>

## 9. Look at the clips, rank the drawings again, export the adapter and reload it

Six held-out clips are written as panels (`outputs/xclip_video_classification_examples/`: the 8-frame strip with the reference label, the frozen top-3 and the adapted top-3 beneath it) so the numbers can be checked by eye. The five drawn clips from Section 5 are then ranked again by the adapted model — the fusion head that was tuned ranks every request, so this is a small look at what the adaptation did *outside* its label set and its corpus: the build record measured before adaptation `a ball rolling to the right` → `a ball standing still` (0.69); `a ball bouncing up and down` → `a ball standing still` (0.51); `a square growing larger` → `a ball standing still` (0.43); `the sun setting` → `a ball standing still` (0.46); `a ball standing still` → `a ball standing still` (0.57) (top-1 0.20 over the five drawn labels); after adaptation `a ball rolling to the right` → `a ball standing still` (0.67); `a ball bouncing up and down` → `a ball standing still` (0.47); `a square growing larger` → `a ball bouncing up and down` (0.37); `the sun setting` → `a ball standing still` (0.41); `a ball standing still` → `a ball standing still` (0.55) (top-1 0.20) — five cartoons of evidence, not a measurement.

`pipe.save_artifact` writes the trained tensors — the fusion head, about 41 MB in float32 — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the label set the head was trained on, the training configuration and the epoch history (OUT8). `XClipVideoClassificationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the fusion head, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical rankings on eight test clips (VER4).

In [ ]:
import shutil

examples_dir = Path('outputs/xclip_video_classification_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
caption_font = ImageFont.load_default(size=18)
for record, frozen_ranking, adapted_ranking in zip(test_records[:6], frozen_test['rankings'][:6], adapted_test['rankings'][:6], strict=True):
    strip = frame_strip(record['frames'], height=120)
    sheet = Image.new('RGB', (max(strip.width, 1400), strip.height + 96), (255, 255, 255))
    sheet.paste(strip, (0, 0))
    marker = ImageDraw.Draw(sheet)
    for i, (tag, text) in enumerate((('REF', record['label']), ('FROZEN', ' > '.join(frozen_ranking[:3])), ('ADAPTED', ' > '.join(adapted_ranking[:3])))):
        marker.text((8, strip.height + 6 + i * 28), f'{tag}: {text[:140]}', fill=(20, 20, 20) if tag != 'FROZEN' else (150, 40, 40), font=caption_font)
    sheet.save(examples_dir / f"{record['id']}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'rows': ['reference', 'frozen top-3', 'adapted top-3']})

adapted_drawn, adapted_drawn_timings, adapted_drawn_checks, adapted_drawn_report = rank_drawings(pipe, 'adapted')

artifact_dir = Path('outputs/xclip_video_classification_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'xclip_video_classification', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'labels': artifact_manifest['adapter']['labels'], 'best_epoch': artifact_manifest['adapter']['best_epoch']})

reloaded = XClipVideoClassificationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [[p['label'] for p in item['predictions']] for item in pipe.classify_batch([r['frames'] for r in test_records[:8]], LABELS)]
after = [[p['label'] for p in item['predictions']] for item in reloaded.classify_batch([r['frames'] for r in test_records[:8]], LABELS)]
parity = {'identical_rankings': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_rankings'] == parity['of']  # a contract check: the artifact reloads ranking-for-ranking

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'labels': LABELS,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'license': CORPUS_LICENSE, 'row_groups': sorted(ROW_GROUP_PINS), 'shard_bytes': CORPUS_BYTES, 'classes': list(SAMPLE_CLASS_TEXT)},
    'inference_contract': {'input_manifest': input_manifest, 'drawn': {'names': drawn_names, 'labels': DRAWN_LABELS, 'digests': drawn_digests}, 'frozen': {'results': frozen_drawn, 'seconds': frozen_drawn_timings, 'checks': frozen_drawn_checks, 'report': frozen_drawn_report}, 'adapted': {'results': adapted_drawn, 'seconds': adapted_drawn_timings, 'checks': adapted_drawn_checks, 'report': adapted_drawn_report}, 'output_files': ['outputs/xclip_video_classification_drawing_frozen.json', 'outputs/xclip_video_classification_drawing_adapted.json']},
    'comparison': comparison,
    'examples': 'outputs/xclip_video_classification_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'av': av.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/xclip_video_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

{'examples': ['hmdb51-test-0-10.png', 'hmdb51-test-0-11.png', 'hmdb51-test-0-12.png', 'hmdb51-test-0-23.png', 'hmdb51-test-0-28.png', 'hmdb51-test-0-29.png'], 'rows': ['reference', 'frozen top-3', 'adapted top-3']}


{'adapted': {'seconds': [0.052, 0.052, 0.051, 0.055, 0.054], 'checks': {'probabilities_sum_to_one': True, 'one_entry_per_label': True, 'ranked_descending': True, 'identity_reported': True}, 'top1': [('synthetic_a_ball_rolling_to_the_right', 'a ball standing still', 0.667), ('synthetic_a_ball_bouncing_up_and_down', 'a ball standing still', 0.468), ('synthetic_a_square_growing_larger', 'a ball bouncing up and down', 0.367), ('synthetic_the_sun_setting', 'a ball standing still', 0.405), ('synthetic_a_ball_standing_still', 'a ball standing still', 0.549)], 'verdict': 'sample-sanity', 'top1_accuracy': 0.2, 'chance': 0.2}}
{'artifact': 'outputs/xclip_video_classification_adapter', 'format': 'org.valcorza.xclip-base-patch32.adapter.v1', 'tensors': 50, 'bytes': 40996184, 'sha256': '5129c534576354e5...', 'labels': ['brushing hair', 'doing a cartwheel', 'catching a ball', 'chewing', 'clapping hands', 'climbing', 'climbing stairs', 'diving into water', 'drawing a sword', 'dribbling a basketball']

{'reload_parity': {'identical_rankings': 8, 'of': 8}, 'reloaded_best_epoch': 2}
['xclip_video_classification_adapter', 'xclip_video_classification_drawing_adapted.json', 'xclip_video_classification_drawing_frozen.json', 'xclip_video_classification_evaluation_report.json', 'xclip_video_classification_example_clip.png', 'xclip_video_classification_examples', 'xclip_video_classification_input_manifest.json', 'xclip_video_classification_result.json', 'xclip_video_classification_train.csv']


**What to notice:** the six panels in `outputs/xclip_video_classification_examples/`, the drawn clips' rankings before and after, and the reload parity line.

<details><summary>Check your reasoning</summary>In the release run the drawn clips stayed almost all `a ball standing still` before and after adaptation — the tuned head was never shown cartoons — with one ranking changed. Reload parity held: eight of eight identical rankings.</details>

## 10. Change one thing: a ten-times-higher learning rate (optional)

*Evaluation practice.* A **Predict → Change one thing → Run → Observe → Explain** activity, off by default so Run all is unaffected. Set `RUN_EXPERIMENT = True`, change **one** field — by default the learning rate goes from 1e-5 to 1e-4 — and run this cell after Sections 4–9. The experiment loads its **own** pipeline from the verified snapshot, so it starts from the checkpoint and never touches the default `pipe`; it writes only to `outputs/xclip_video_classification_experiment/`, prints the default and the changed run side by side (epoch 0 must match), and checks that the default exports are byte-identical afterwards.

**Predict:** at 1e-4, at which epoch will validation top-1 peak?

In [ ]:
RUN_EXPERIMENT = False  # @param {type:"boolean"}
EXPERIMENT_LEARNING_RATE = 1e-4  # @param {type:"number"}
EXPERIMENT_EPOCHS = 8  # @param {type:"integer"}

if not RUN_EXPERIMENT:
    print({'experiment': 'skipped (RUN_EXPERIMENT = False); the default path above is complete'})
else:
    canonical_files = {'adapter': artifact_dir / 'adapter.safetensors', 'evaluation_report': Path('outputs/xclip_video_classification_evaluation_report.json'), 'result': Path('outputs/xclip_video_classification_result.json')}
    canonical = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in canonical_files.items()}
    experiment_dir = Path('outputs/xclip_video_classification_experiment')
    shutil.rmtree(experiment_dir, ignore_errors=True)
    experiment_dir.mkdir(parents=True)
    # Its own pipeline from the verified snapshot: the experiment starts from the checkpoint and the default pipe is untouched.
    experiment_pipe = XClipVideoClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)
    experiment_result = experiment_pipe.adapt(train_records, val_records, LABELS, epochs=EXPERIMENT_EPOCHS, lr=EXPERIMENT_LEARNING_RATE, batch_size=BATCH_SIZE, progress=report)
    experiment_test = experiment_pipe.evaluate(test_records, LABELS, batch_size=EVAL_BATCH_SIZE)
    side_by_side = {
        'settings': {'default': {'lr': adapt_result['lr'], 'epochs': adapt_result['epochs']}, 'experiment': {'lr': EXPERIMENT_LEARNING_RATE, 'epochs': EXPERIMENT_EPOCHS}},
        'validation_top1_by_epoch': {'default': [round((h['val'] or {}).get('top1_accuracy', float('nan')), 3) for h in adapt_result['history']], 'experiment': [round((h['val'] or {}).get('top1_accuracy', float('nan')), 3) for h in experiment_result['history']]},
        'best_epoch': {'default': adapt_result['best_epoch'], 'experiment': experiment_result['best_epoch']},
        'test': {metric: {'frozen': round(frozen_test[metric], 3), 'default': round(adapted_test[metric], 3), 'experiment': round(experiment_test[metric], 3)} for metric in METRICS},
    }
    for key, row in side_by_side.items():
        print({key: row})
    with open(experiment_dir / 'experiment_report.json', 'w', encoding='utf-8') as handle:
        json.dump({'side_by_side': side_by_side, 'history': experiment_result['history']}, handle, indent=2, ensure_ascii=False, default=str)
    unchanged = {name: hashlib.sha256(path.read_bytes()).hexdigest() == canonical[name] for name, path in canonical_files.items()}
    if not all(unchanged.values()):
        raise RuntimeError(f'the experiment changed a default export: {unchanged}')
    print({'default_exports_unchanged': unchanged, 'experiment_outputs': str(experiment_dir)})
    del experiment_pipe

{'experiment': 'skipped (RUN_EXPERIMENT = False); the default path above is complete'}


**Observe → Explain.** Compare the two validation curves (epoch 0 must be equal) and the `test` rows.

<details><summary>Check your reasoning</summary>The build record found that at 1e-4 a head this size memorises 181 clips within an epoch or two, so validation top-1 peaks at epoch 1 and the curve is then flat; the earliest best epoch is kept. A faster climb is not a better model: compare the test rows, and read a difference of one or two clips (1.5 points each) as noise. No experiment run is recorded on the release runtime.</details>

## Interpretation and limits

A video–text model trained on Kinetics-400 — whose vocabulary already covers most of these ten actions — ranks the right one of ten HMDB51 actions first for 0.803 of the held-out clips in the Kaggle T4 release run (21 September 2026); a bounded fine-tuning of its fusion head on 181 clips moves that to 0.909 top-1 and 0.905 macro F1 in the build record, with a 41 MB adapter that reloads ranking-for-ranking. That is the claim: the adaptation contract works end to end on a closed label set with a real labelled set, and the numbers it produces are read as top-1 / top-3 accuracy, macro recall and macro F1 against two non-adapted baselines and the frozen model, with the per-label recall beside them rather than in isolation. The Tesla T4 run reproduced the CPU sweep's numbers exactly (0.909 / 0.985 / 0.905 at epoch 2). This row has no sibling on its corpus — X-CLIP is the fleet's only video model — so the comparison is the frozen prompts against the tuned head on the same 66 clips: the gain comes from four labels (`chewing` 0.43 → 0.86, `diving into water` 0.67 → 1.00, `drawing a sword` 0.71 → 1.00, `clapping hands` 0.50 → 0.67) while `dribbling a basketball` lost one clip (0.71 → 0.57), and the five drawn cartoon clips stay at chance before and after (every clip but one is called `a ball standing still`): the head learned the ten HMDB51 actions, not motion in general.

The test split is 66 clips from one seeded, source-grouped draw of one 300-clip sample, the validation split that picks the epoch is 53, and every rate is over one reference label per clip — not a benchmark, not the HMDB51 protocol (three splits over all 51 classes), not a measure of temporal localisation. So a result here says the contract works on ten actions cut from films, not that the adapted model handles other actions, other cameras or your clips. The head that was tuned ranks every request: the drawn clips re-ranked in Section 9 are five cartoons of evidence about what the tuning did outside its label set (before adaptation `a ball rolling to the right` → `a ball standing still` (0.69); `a ball bouncing up and down` → `a ball standing still` (0.51); `a square growing larger` → `a ball standing still` (0.43); `the sun setting` → `a ball standing still` (0.46); `a ball standing still` → `a ball standing still` (0.57) (top-1 0.20 over the five drawn labels); after adaptation `a ball rolling to the right` → `a ball standing still` (0.67); `a ball bouncing up and down` → `a ball standing still` (0.47); `a square growing larger` → `a ball bouncing up and down` (0.37); `the sun setting` → `a ball standing still` (0.41); `a ball standing still` → `a ball standing still` (0.55) (top-1 0.20)), not a measurement, and a deployment that ranks other label sets must measure them after adapting. The towers were not adapted: what the frame encoder cannot see stays unseen, and **the probabilities remain a softmax over your label set**.

Three things to carry to real data. **Baselines first:** the chance and majority rates on *your* labels, and the frozen model's per-label recall, are the numbers to read before any adapted one. **Macro over micro:** a class the model never predicts costs a full tenth of macro recall while barely moving top-1 on a skewed set; read both. **Leakage:** keep every clip in one split (the contract de-duplicates by decoded pixels) and split by source video, camera or session when your clips come from few recordings — clips cut from one video share its scene and actor.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled clip set, validate the demonstrated dataset contract without leakage, execute the inference contract for five clips and a bounded fine-tuning of the fusion head with the closed-set cross-entropy, evaluate against two non-adapted baselines and the frozen model on a source-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, classification quality on any other action set or video domain, ranking quality on other label sets after adaptation, or production fitness.

**Optional experiments (off by default; each names its field and what to run):** Section 10 runs a ten-times-higher learning rate in its own pipeline and prints it beside the default run — change `EXPERIMENT_LEARNING_RATE` or `EXPERIMENT_EPOCHS` there and run that cell again. Changing `EPOCHS` or `LEARNING_RATE` and choosing **Run after** from Section 7, or `SPLIT_SEED` and **Run after** from Section 4, also starts from the pinned base — every `adapt` and Section 4 put it back first — but replaces the default results and exports. Editing `SAMPLE_CLASS_TEXT`'s phrasing in the carried module and running from Section 4 shows how much the pre-trained model depends on the wording of a label. BYOD: `USE_BYOD` and `BYOD_PATH` in Section 4, then **Run after** from Section 4, and read the two baselines before the adapted number.

## Troubleshooting

- **Section 1 stops with "This notebook needs a Linux x86_64 runtime"** — you are on Windows, macOS or an ARM machine. Use Google Colab, Kaggle or a Linux x86_64 Jupyter server.
- **The uv wheel fails its size/SHA-256 check, or a download in Section 1 times out** — run Section 1 again; a complete environment is reused, an incomplete one is finished. If it repeats, the network is blocking or altering `files.pythonhosted.org` or `pypi.org`.
- **"The isolated environment's Python process exited"** — usually out of memory. Restart the session and choose **Run all**; leave the optional experiment off on a small runtime.
- **You re-ran Section 1 on its own** — nothing is lost: it keeps the running worker and every variable, so the cells after it keep working. After a session restart, run from the top.
- **Section 3 reports a size or SHA-256 mismatch, or cannot reach the Hub** — the message names the file. Delete it from the snapshot folder Section 3 prints and run Section 3 again; the snapshot comes from `huggingface.co`.
- **Section 4 names the parquet row group in a `sha256` error** — the cached `weights/hmdb51/test-rg0.parquet` is incomplete; delete it and run Section 4 again (the default path needs `huggingface.co`).
- **Out of memory** — lower `BATCH_SIZE` in Section 7, or restart the session and choose **Run all**; leave Section 10 off on a small runtime.
- **BYOD: "BYOD_PATH … does not exist"** — the path is relative to the working directory printed in the message.
- **BYOD: "the upload dialog exists only in Google Colab"** — on Kaggle or Jupyter, put the zip in the runtime (or attach it as a dataset) and set `BYOD_PATH`.
- **BYOD: "Upload exactly one .zip file"** — the dialog was cancelled or several files were chosen; run the cell again.
- **BYOD: "labels.csv line N (file …): names a missing clip"** — fix the `file` column of that row, or add the clip to the zip.
- **BYOD: "… not a decodable clip of at least 8 frames"** — the file on that line is corrupt, too short, or in a container PyAV cannot read.
- **BYOD: "the train split holds …" or "the … split holds no clip of …"** — add clips or sources; the message names the minimum for your label count.
## Glossary

- **Cross-frame attention** — each frame's encoder attends to the other frames, so motion informs every frame embedding.
- **Frame-integration transformer** — a one-layer transformer that fuses the 8 frame embeddings into one video embedding.
- **Prompt generator** — conditions the class-name embeddings on the clip's patch features.
- **Fusion head** — the integration transformer, the visual projections and the prompt generator: the only trained part.
- **Frozen-tower cache** — the frozen towers' outputs computed once and reused, so each step runs only the head.
- **Top-k accuracy** — the reference label is among the k highest-ranked names.
- **Macro recall / macro F1** — the mean per-label recall or F1, so a never-predicted label costs a full share.
- **Source grouping** — clips cut from one source video stay in one split.
- **Kinetics-400 overlap** — labels whose action the model was trained on; the pre-trained score on them is not zero-shot transfer.
- **Adapter / reload parity** — the trained tensors only (safetensors) overlaid on the pinned base; the reloaded pipeline ranks identically.
- **BYOD** — bring your own data: your labelled clips through the same cells.

## Conclusion (your notes)

Optional — fill in from **your** run, not the recorded one:

- The task was ___ labels on ___ test clips from ___ sources.
- Labels with a Kinetics-400 counterpart: ___; without: ___.
- Pre-trained top-1 ___ (weakest label ___ at ___); chance ___, majority ___.
- After fine-tuning (epoch ___ kept): top-1 ___, macro F1 ___; labels that moved: ___.
- What I would need before claiming the fine-tune helps on my clips: ___ (for example labels outside Kinetics-400, more test clips, several seeds).

## References

- Repository README: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/xclip-base-patch32
- Upstream code: https://github.com/microsoft/VideoX/tree/master/X-CLIP
- Expanding Language-Image Pretrained Models for General Video Recognition (Ni et al., ECCV 2022): https://arxiv.org/abs/2208.02816
- HMDB51 (Serre Lab, CC BY 4.0): https://huggingface.co/datasets/Serrelab/hmdb51 — Kuehne, Jhuang, Garrote, Poggio, Serre, HMDB: A Large Video Database for Human Motion Recognition (ICCV 2011); parquet repack read here: https://huggingface.co/datasets/mteb/HMDB51
- DIMER Notebook Specification 2.2 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)